# Import LIbraries and Load Datasets

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
# Suppress irrelevant Jupyter datetime deprecation warnings
import warnings

warnings.filterwarnings(
    "ignore",
    category=DeprecationWarning,
    module="jupyter_client"
)

# Colab upload directory
DATA_DIR = Path("/content")

# Exact source filenames
DATASET_FILES = {
    "ball_by_ball": "ball_by_ball.csv",
    "playing_eleven": "playing_eleven.csv",
    "matches": "matches.csv",
    "competition_team_player_mapping": "competition_team_player_mapping.csv",
    "teams": "teams.csv",
    "competitions": "competitions.csv"
}

# Verify that all six files exist before loading
missing_files = [
    filename
    for filename in DATASET_FILES.values()
    if not (DATA_DIR / filename).exists()
]

if missing_files:
    raise FileNotFoundError(
        f"Missing CSV files in {DATA_DIR}:\n{missing_files}"
    )

# Load original datasets
raw_data = {
    name: pd.read_csv(DATA_DIR / filename, low_memory=False)
    for name, filename in DATASET_FILES.items()
}

# Create convenient references to the original DataFrames
df_ball_by_ball_raw = raw_data["ball_by_ball"]
df_playing_eleven_raw = raw_data["playing_eleven"]
df_matches_raw = raw_data["matches"]
df_mapping_raw = raw_data["competition_team_player_mapping"]
df_teams_raw = raw_data["teams"]
df_competitions_raw = raw_data["competitions"]

print("All six source datasets loaded successfully.")
print("Original datasets preserved without cleaning or modification.")


All six source datasets loaded successfully.
Original datasets preserved without cleaning or modification.


In [2]:
inventory = []

for name, df in raw_data.items():
    inventory.append({
        "Dataset": name,
        "Rows": df.shape[0],
        "Columns": df.shape[1],
        "Total Missing Values": int(df.isna().sum().sum()),
        "Exact Duplicate Rows": int(df.duplicated().sum()),
        "Memory Usage (MB)": round(
            df.memory_usage(deep=True).sum() / (1024 ** 2), 2
        )
    })

df_inventory = pd.DataFrame(inventory)

display(df_inventory)

print("\nTotal source rows:",
      df_inventory["Rows"].sum())

print("Total source columns:",
      df_inventory["Columns"].sum())


,Dataset,Rows,Columns,Total Missing Values,Exact Duplicate Rows,Memory Usage (MB)
0,ball_by_ball,200999,33,766466,0,165.28
1,playing_eleven,28381,9,26012,0,3.21
2,matches,883,22,2736,0,0.46
3,competition_team_player_mapping,6413,6,0,0,1.39
4,teams,353,3,0,0,0.03
5,competitions,141,4,0,0,0.03



Total source rows: 237170
Total source columns: 77


# Schema and column-level audit

In [3]:
schema_records = []

for dataset_name, df in raw_data.items():

    for column in df.columns:

        schema_records.append({
            "Dataset": dataset_name,
            "Column": column,
            "Data Type": str(df[column].dtype),
            "Non-Null Count": int(df[column].notna().sum()),
            "Missing Count": int(df[column].isna().sum()),
            "Missing Percentage": round(
                df[column].isna().mean() * 100, 2
            ),
            "Unique Values": int(df[column].nunique())
        })

df_schema = pd.DataFrame(schema_records)

# Display complete schema
display(
    df_schema.sort_values(
        ["Dataset", "Column"]
    ).reset_index(drop=True)
)

print("Total audited columns:", len(df_schema))

,Dataset,Column,Data Type,Non-Null Count,Missing Count,Missing Percentage,Unique Values
0,ball_by_ball,batting_order,int64,200999,0,0.0,13
1,ball_by_ball,batting_team_id,int64,200999,0,0.0,38
2,ball_by_ball,batting_type,object,200999,0,0.0,2
3,ball_by_ball,bowl_speed,float64,200999,0,0.0,1033
4,ball_by_ball,bowler_id,int64,200999,0,0.0,647
...,...,...,...,...,...,...,...
72,playing_eleven,player_id,int64,28381,0,0.0,1082
73,playing_eleven,team_id,int64,28381,0,0.0,38
74,teams,comp_id,int64,353,0,0.0,141
75,teams,team_id,int64,353,0,0.0,38


Total audited columns: 77


In [4]:
missing_profile = (
    df_schema[df_schema["Missing Count"] > 0]
    .sort_values(
        ["Dataset", "Missing Count"],
        ascending=[True, False]
    )
    .reset_index(drop=True)
)

display(missing_profile)

print(
    "\nColumns containing missing values:",
    len(missing_profile)
)

print(
    "Total missing values:",
    missing_profile["Missing Count"].sum()
)

,Dataset,Column,Data Type,Non-Null Count,Missing Count,Missing Percentage,Unique Values
0,ball_by_ball,dismissal_type_id,float64,10881,190118,94.59,8
1,ball_by_ball,dismissal_type,object,10881,190118,94.59,9
2,ball_by_ball,dismissed_player_id,float64,10892,190107,94.58,814
3,ball_by_ball,extras_type,object,11832,189167,94.11,6
4,ball_by_ball,stroke,object,194841,6158,3.06,30
5,ball_by_ball,over_or_around,object,200592,407,0.20,2
6,ball_by_ball,delivery_style,object,200608,391,0.19,34
7,matches,revised_target,float64,23,860,97.40,23
8,matches,first_inn_revised_overs,float64,25,858,97.17,13
9,matches,second_inn_revised_overs,float64,25,858,97.17,12



Columns containing missing values: 18
Total missing values: 795214


### Investigate the 11 additional missing batting orders

In [5]:
df_xi = df_playing_eleven_raw

print("is_played value counts:")
display(
    df_xi["is_played"].value_counts(dropna=False)
)

# Missing batting order by playing status
print("\nMissing batting_order by is_played:")

display(
    df_xi.groupby("is_played", dropna=False)
    .agg(
        Total_Records=("player_id", "size"),
        Missing_Batting_Order=("batting_order",
                               lambda x: x.isna().sum()),
        Missing_Captain=("is_captain",
                         lambda x: x.isna().sum()),
        Missing_WK=("is_wk",
                    lambda x: x.isna().sum())
    )
)

# Inspect records that are marked as played but lack batting order
played_missing_order = df_xi[
    (df_xi["is_played"] == True) &
    (df_xi["batting_order"].isna())
].copy()

print("\nPlayed records with missing batting order:")
print("Count:", len(played_missing_order))

display(played_missing_order)

# Inspect non-playing records with batting order populated
unplayed_with_order = df_xi[
    (df_xi["is_played"] == False) &
    (df_xi["batting_order"].notna())
].copy()

print("\nNon-playing records with batting order populated:")
print("Count:", len(unplayed_with_order))

display(unplayed_with_order)

is_played value counts:


,count
is_played,
True,19714
False,8667



Missing batting_order by is_played:


,Total_Records,Missing_Batting_Order,Missing_Captain,Missing_WK
is_played,,,,
False,8667,8667,8667,8667
True,19714,11,0,0



Played records with missing batting order:
Count: 11


,match_id,team_id,player_id,batting_order,is_played,is_captain,is_wk,is_impact,is_impact_off
22883,48512,25,7690,NaN,True,False,False,0,0
22884,48512,25,3091,NaN,True,False,False,0,0
22885,48512,25,20327,NaN,True,False,True,0,0
22886,48512,25,7696,NaN,True,False,False,0,0
22887,48512,25,20704,NaN,True,False,False,0,0
22888,48512,25,2921,NaN,True,False,False,0,0
22889,48512,25,2490,NaN,True,False,False,0,0
22890,48512,25,5979,NaN,True,False,False,0,0
22891,48512,25,4926,NaN,True,False,False,0,0
22892,48512,25,5189,NaN,True,False,False,0,0



Non-playing records with batting order populated:
Count: 0


,match_id,team_id,player_id,batting_order,is_played,is_captain,is_wk,is_impact,is_impact_off


### Investigate match 48512

In [6]:
MATCH_ID = 48512
TEAM_ID = 25

# 1. Inspect match metadata
match_details = df_matches_raw[
    df_matches_raw["match_id"] == MATCH_ID
]

print("MATCH METADATA")
display(match_details)

# 2. Inspect all playing XI registrations for this team
match_xi = df_playing_eleven_raw[
    (df_playing_eleven_raw["match_id"] == MATCH_ID) &
    (df_playing_eleven_raw["team_id"] == TEAM_ID)
].copy()

print("\nPLAYING XI RECORDS")
display(match_xi.sort_values("batting_order", na_position="last"))

# 3. Inspect delivery columns
print("\nDELIVERY PLAYER/TEAM COLUMNS")

print([
    col for col in df_ball_by_ball_raw.columns
    if any(term in col.lower() for term in
           ["player", "striker", "bowler", "batter", "team", "innings"])
])

# 4. Inspect all deliveries in the match
match_deliveries = df_ball_by_ball_raw[
    df_ball_by_ball_raw["match_id"] == MATCH_ID
].copy()

print("\nMATCH DELIVERY SUMMARY")
print("Total deliveries:", len(match_deliveries))

display(
    match_deliveries.groupby("innings")
    .agg(
        Delivery_Records=("match_id", "size"),
        Unique_Batters=("striker_id", "nunique"),
        Unique_Non_Strikers=("non_striker_id", "nunique"),
        Unique_Bowlers=("bowler_id", "nunique")
    )
)

# 5. Identify delivery participation for the 11 affected players
affected_players = match_xi["player_id"].tolist()

role_columns = [
    "striker_id",
    "non_striker_id",
    "bowler_id"
]

participation_records = []

for player_id in affected_players:

    record = {
        "player_id": player_id
    }

    for role in role_columns:
        record[role] = (
            match_deliveries[role] == player_id
        ).sum()

    record["Total_Delivery_Roles"] = sum(
        record[role] for role in role_columns
    )

    participation_records.append(record)

df_affected_participation = pd.DataFrame(
    participation_records
)

print("\nDELIVERY PARTICIPATION OF AFFECTED PLAYERS")
display(df_affected_participation)

# 6. Check whether the match belongs to the approved population
print("\nMATCH STATUS")
display(
    match_details[[
        col for col in [
            "match_id", "comp_id", "match_result"
        ] if col in match_details.columns
    ]]
)

MATCH METADATA


,match_id,comp_id,match_number,match_date,stage,venue_id,venue_country,home_team_id,home_team,away_team_id,...,away_team_captain_id,toss_win_team_id,toss_opted,man_of_the_match,first_inn_revised_overs,second_inn_revised_overs,revised_target,knock_out,win_team_id,result
824,48512,3317,1,2026-01-07,Others,60,Sri Lanka,27,Sri Lanka,25,...,3489.0,25.0,Fielding,4926,NaN,NaN,NaN,0,25.0,Pakistan Won by 6 wickets



PLAYING XI RECORDS


,match_id,team_id,player_id,batting_order,is_played,is_captain,is_wk,is_impact,is_impact_off
22879,48512,25,37879,NaN,False,NaN,NaN,0,0
22880,48512,25,6922,NaN,False,NaN,NaN,0,0
22881,48512,25,52338,NaN,False,NaN,NaN,0,0
22882,48512,25,8049,NaN,False,NaN,NaN,0,0
22883,48512,25,7690,NaN,True,False,False,0,0
22884,48512,25,3091,NaN,True,False,False,0,0
22885,48512,25,20327,NaN,True,False,True,0,0
22886,48512,25,7696,NaN,True,False,False,0,0
22887,48512,25,20704,NaN,True,False,False,0,0
22888,48512,25,2921,NaN,True,False,False,0,0



DELIVERY PLAYER/TEAM COLUMNS
['innings', 'innings_ball_num', 'batting_team_id', 'bowling_team_id', 'striker_id', 'striker_name', 'non_striker_id', 'non_striker_name', 'bowler_id', 'bowler_name', 'dismissed_player_id']

MATCH DELIVERY SUMMARY
Total deliveries: 219


,Delivery_Records,Unique_Batters,Unique_Non_Strikers,Unique_Bowlers
innings,,,,
1,116,11,11,6
2,103,6,6,6



DELIVERY PARTICIPATION OF AFFECTED PLAYERS


,player_id,striker_id,non_striker_id,bowler_id,Total_Delivery_Roles
0,37879,0,0,0,0
1,6922,0,0,0,0
2,52338,0,0,0,0
3,8049,0,0,0,0
4,7690,19,17,0,36
5,3091,36,37,0,73
6,20327,13,17,0,30
7,7696,0,0,14,14
8,20704,0,0,24,24
9,2921,11,8,0,19



MATCH STATUS


,match_id,comp_id
824,48512,3317


### Record the finding in the exception register

In [7]:
phase2_exceptions = pd.DataFrame([
    {
        "Exception_ID": "P2-XI-001",
        "Dataset": "playing_eleven",
        "Match_ID": 48512,
        "Affected_Records": 11,
        "Field": "batting_order",
        "Observation": (
            "11 played registrations have missing batting_order. "
            "All 11 have delivery-level participation."
        ),
        "Treatment": (
            "Preserve missing values. Do not infer or impute "
            "batting_order. Use delivery-level batting_order "
            "where available."
        ),
        "Status": "Documented - Unresolved"
    }
])

display(phase2_exceptions)


,Exception_ID,Dataset,Match_ID,Affected_Records,Field,Observation,Treatment,Status
0,P2-XI-001,playing_eleven,48512,11,batting_order,11 played registrations have missing batting_o...,Preserve missing values. Do not infer or imput...,Documented - Unresolved


In [8]:
df_bb = df_ball_by_ball_raw

# Inspect relevant columns
run_columns = [
    "runs",
    "extras1_runs",
    "extras_2_runs",
    "extras_type",
    "is_four",
    "is_six",
    "is_wicket",
    "dismissal_type",
    "dismissed_player_id"
]

print("DATA TYPES")
display(df_bb[run_columns].dtypes.to_frame("Data Type"))

# 1. Runs distribution
print("\nRUNS DISTRIBUTION")
display(
    df_bb["runs"]
    .value_counts(dropna=False)
    .sort_index()
    .to_frame("Delivery_Count")
)

# 2. Encoded extras category distribution
print("\nEXTRAS1_RUNS DISTRIBUTION")
display(
    df_bb["extras1_runs"]
    .value_counts(dropna=False)
    .sort_index()
    .to_frame("Delivery_Count")
)

# 3. Extras_2_runs distribution
print("\nEXTRAS_2_RUNS DISTRIBUTION")
display(
    df_bb["extras_2_runs"]
    .value_counts(dropna=False)
    .sort_index()
    .to_frame("Delivery_Count")
)

# 4. Extras type distribution
print("\nEXTRAS_TYPE DISTRIBUTION")
display(
    df_bb["extras_type"]
    .value_counts(dropna=False)
    .to_frame("Delivery_Count")
)

# 5. Boundary flag distribution
print("\nBOUNDARY FLAG COMBINATIONS")
display(
    df_bb.groupby(
        ["runs", "is_four", "is_six"],
        dropna=False
    ).size()
    .reset_index(name="Delivery_Count")
    .sort_values("Delivery_Count", ascending=False)
)

# 6. Wicket distribution
print("\nWICKET DISTRIBUTION")
display(
    df_bb.groupby(
        ["is_wicket", "dismissal_type"],
        dropna=False
    ).size()
    .reset_index(name="Delivery_Count")
    .sort_values("Delivery_Count", ascending=False)
)

DATA TYPES


,Data Type
runs,int64
extras1_runs,int64
extras_2_runs,int64
extras_type,object
is_four,bool
is_six,bool
is_wicket,bool
dismissal_type,object
dismissed_player_id,float64



RUNS DISTRIBUTION


,Delivery_Count
runs,
0,79387
1,72267
2,12547
3,656
4,23303
5,25
6,12814



EXTRAS1_RUNS DISTRIBUTION


,Delivery_Count
extras1_runs,
1,576
2,7714
3,2748
4,529
5,264
6,189167
7,1


... (4 outputs left out)


BOUNDARY FLAG COMBINATIONS


,runs,is_four,is_six,Delivery_Count
0,0,False,False,79387
1,1,False,False,72267
5,4,True,False,23283
8,6,False,True,12811
2,2,False,False,12547
3,3,False,False,656
6,5,False,False,25
4,4,False,False,20
7,6,False,False,3



WICKET DISTRIBUTION


,is_wicket,dismissal_type,Delivery_Count
0,False,NaN,190118
2,True,Caught,7557
1,True,Bowled,1775
4,True,LBW,709
8,True,RunOut,600
9,True,Stump,214
3,True,Hit Wicket,17
7,True,Retire Out,6
5,True,Obstracting the field,2
6,True,Obstructing the field,1


### Investigate the 36 non-zero extras_2_runs records

In [9]:
df_bb = df_ball_by_ball_raw

# Select all records with non-zero extras_2_runs
extras2_exceptions = df_bb[
    df_bb["extras_2_runs"] != 0
].copy()

print("NON-ZERO EXTRAS_2_RUNS RECORDS")
print("Total:", len(extras2_exceptions))

inspection_columns = [
    "match_id",
    "innings",
    "innings_ball_num",
    "overid",
    "over_ball_num",
    "batting_team_id",
    "bowling_team_id",
    "striker_id",
    "bowler_id",
    "runs",
    "extras1_runs",
    "extras_2_runs",
    "extras_type",
    "is_four",
    "is_six",
    "is_wicket",
    "dismissal_type"
]

display(
    extras2_exceptions[inspection_columns]
    .sort_values(["match_id", "innings", "innings_ball_num"])
    .reset_index(drop=True)
)

# Distribution by extras category
print("\nEXTRAS_2_RUNS BY EXTRAS TYPE")

display(
    extras2_exceptions.groupby(
        ["extras1_runs", "extras_type", "extras_2_runs"],
        dropna=False
    ).size()
    .reset_index(name="Delivery_Count")
    .sort_values(
        ["extras1_runs", "extras_2_runs"]
    )
)

# Cross-tabulation with runs
print("\nEXTRAS_2_RUNS VS RUNS")

display(
    pd.crosstab(
        extras2_exceptions["extras_2_runs"],
        extras2_exceptions["runs"],
        margins=True
    )
)

NON-ZERO EXTRAS_2_RUNS RECORDS
Total: 36


,match_id,innings,innings_ball_num,overid,over_ball_num,batting_team_id,bowling_team_id,striker_id,bowler_id,runs,extras1_runs,extras_2_runs,extras_type,is_four,is_six,is_wicket,dismissal_type
0,13523,1,4,1,4,18,28,4117,2417,0,1,4,Overstep,False,False,False,NaN
1,13614,1,111,18,5,27,20,2436,829,0,5,4,other,False,False,False,NaN
2,14112,1,66,11,5,7,8,220,6256,0,1,4,Overstep,False,False,False,NaN
3,14127,2,52,9,3,7,3,5928,4837,0,5,4,other,False,False,False,NaN
4,14159,2,51,9,1,2,242,4117,25048,0,5,1,other,False,False,False,NaN
5,15796,2,88,14,4,23,18,4814,25048,0,1,1,Overstep,False,False,False,NaN
6,16424,2,33,6,1,21,72,2218,21786,0,1,4,Overstep,False,False,False,NaN
7,17555,1,29,5,4,22,25,728,7364,0,6,1,NaN,False,False,False,NaN
8,20186,2,130,20,2,3509,1,914,6806,0,1,1,Overstep,False,False,False,NaN
9,20188,1,117,19,4,4,3,3782,6591,0,5,1,other,False,False,False,NaN



EXTRAS_2_RUNS BY EXTRAS TYPE


,extras1_runs,extras_type,extras_2_runs,Delivery_Count
0,1,Overstep,1,12
1,1,Overstep,4,9
2,5,other,1,7
3,5,other,4,7
4,6,NaN,1,1



EXTRAS_2_RUNS VS RUNS


runs,0,All
extras_2_runs,,
1,20,20
4,16,16
All,36,36


### Inspect the 23 boundary inconsistencies

In [10]:
boundary_exceptions = df_bb[
    (
        (df_bb["runs"] == 4) &
        (~df_bb["is_four"])
    ) |
    (
        (df_bb["runs"] == 6) &
        (~df_bb["is_six"])
    )
].copy()

print("BOUNDARY EXCEPTION COUNT:", len(boundary_exceptions))

display(
    boundary_exceptions[
        inspection_columns
    ].sort_values(
        ["match_id", "innings", "innings_ball_num"]
    ).reset_index(drop=True)
)

# Check whether exceptions involve extras
print("\nBOUNDARY EXCEPTIONS BY EXTRAS CATEGORY")

display(
    boundary_exceptions.groupby(
        ["runs", "extras_type", "extras_2_runs"],
        dropna=False
    ).size()
    .reset_index(name="Delivery_Count")
)

BOUNDARY EXCEPTION COUNT: 23


,match_id,innings,innings_ball_num,overid,over_ball_num,batting_team_id,bowling_team_id,striker_id,bowler_id,runs,extras1_runs,extras_2_runs,extras_type,is_four,is_six,is_wicket,dismissal_type
0,14113,1,3,1,3,3,3509,6314,2399,4,6,0,NaN,False,False,False,NaN
1,14123,1,19,3,6,3,8,813,403,4,6,0,NaN,False,False,False,NaN
2,14149,1,72,12,2,1,7,1941,818,4,6,0,NaN,False,False,False,NaN
3,15609,2,106,17,8,78,72,2533,6523,4,6,0,NaN,False,False,False,NaN
4,16425,2,74,13,1,21,72,5882,4867,4,6,0,NaN,False,False,False,NaN
5,16754,1,33,6,1,78,23,2533,7094,4,6,0,NaN,False,False,False,NaN
6,17396,1,115,19,3,20,18,6396,2933,6,6,0,NaN,False,False,False,NaN
7,20236,2,13,2,7,3,8,6819,29888,4,6,0,NaN,False,False,False,NaN
8,24350,2,38,6,6,18,28,7897,3275,4,6,0,NaN,False,False,False,NaN
9,28958,1,14,3,2,27,72,3669,4874,4,6,0,NaN,False,False,False,NaN



BOUNDARY EXCEPTIONS BY EXTRAS CATEGORY


,runs,extras_type,extras_2_runs,Delivery_Count
0,4,NaN,0,20
1,6,NaN,0,3


### Investigate the relationship between runs and extras

In [11]:
runs_extras_analysis = (
    df_bb.groupby(
        ["extras1_runs", "extras_type"],
        dropna=False
    )
    .agg(
        Delivery_Count=("runs", "size"),
        Distinct_Run_Values=("runs", "nunique"),
        Minimum_Runs=("runs", "min"),
        Maximum_Runs=("runs", "max"),
        Total_Runs_Field=("runs", "sum"),
        NonZero_Extras2=(
            "extras_2_runs",
            lambda x: (x != 0).sum()
        )
    )
    .reset_index()
)

display(runs_extras_analysis)

,extras1_runs,extras_type,Delivery_Count,Distinct_Run_Values,Minimum_Runs,Maximum_Runs,Total_Runs_Field,NonZero_Extras2
0,1,Overstep,576,6,0,6,852,21
1,2,wide,7714,1,0,0,0,0
2,3,legbyes,2748,2,0,1,1,0
3,4,byes,529,1,0,0,0,0
4,5,other,264,6,0,6,412,14
5,6,NaN,189167,7,0,6,268285,1
6,7,Penalty,1,1,0,0,0,0


### Reconstruct the meaning of runs using independent evidence

#### Identify match score fields

In [12]:
print("MATCHES DATASET COLUMNS")

for col in df_matches_raw.columns:
    print(col)

print("\nPOSSIBLE SCORE-RELATED FIELDS")

score_keywords = [
    "run", "score", "wicket", "over",
    "inning", "target", "result"
]

score_columns = [
    col for col in df_matches_raw.columns
    if any(keyword in col.lower() for keyword in score_keywords)
]

display(
    df_matches_raw[score_columns].head(15)
)

MATCHES DATASET COLUMNS
match_id
comp_id
match_number
match_date
stage
venue_id
venue_country
home_team_id
home_team
away_team_id
away_team
home_team_captain_id
away_team_captain_id
toss_win_team_id
toss_opted
man_of_the_match
first_inn_revised_overs
second_inn_revised_overs
revised_target
knock_out
win_team_id
result

POSSIBLE SCORE-RELATED FIELDS


,first_inn_revised_overs,second_inn_revised_overs,revised_target,result
0,NaN,NaN,NaN,West Indies Won by 9 Wickets
1,NaN,NaN,NaN,England Won by 1 Run
2,NaN,NaN,NaN,West Indies Won by 20 runs
3,NaN,NaN,NaN,England Won by 34 runs
4,NaN,NaN,NaN,West Indies Won by 17 runs
5,NaN,NaN,NaN,India Won by 6 Wickets
6,NaN,NaN,NaN,India Won by 8 Runs
7,NaN,NaN,NaN,India Won by 17 runs
8,NaN,114.0,143.0,Australia Won by 20 runs (D/L method)
9,NaN,NaN,NaN,Australia Won the one over Eliminator


#### Compare delivery aggregates by innings

In [13]:
innings_run_summary = (
    df_bb.groupby(
        ["match_id", "innings"],
        as_index=False
    )
    .agg(
        Delivery_Count=("runs", "size"),
        Runs_Sum=("runs", "sum"),
        Extras2_Sum=("extras_2_runs", "sum"),
        Wickets=("is_wicket", "sum"),
        Unique_Batters=("striker_id", "nunique"),
        Unique_Bowlers=("bowler_id", "nunique")
    )
)

display(innings_run_summary.head(20))

print("\nOVERALL AGGREGATES")

display(
    innings_run_summary[
        [
            "Delivery_Count",
            "Runs_Sum",
            "Extras2_Sum",
            "Wickets"
        ]
    ].sum().to_frame("Total")
)

,match_id,innings,Delivery_Count,Runs_Sum,Extras2_Sum,Wickets,Unique_Batters,Unique_Bowlers
0,13511,1,121,100,0,10,10,7
1,13511,2,103,99,0,1,3,6
2,13512,1,128,163,0,8,10,6
3,13512,2,124,158,0,8,10,6
4,13513,1,125,211,0,5,7,6
5,13513,2,127,192,0,9,11,6
6,13514,1,128,182,0,6,7,6
7,13514,2,125,150,0,5,7,6
8,13515,1,127,168,0,4,6,6
9,13515,2,127,150,0,10,10,7



OVERALL AGGREGATES


,Total
Delivery_Count,200999
Runs_Sum,269550
Extras2_Sum,84
Wickets,10881


### Move to legal-delivery reconstruction

#### Reconstruct over-level delivery evidence

In [14]:
# Work on a separate copy
df_legal_audit = df_bb.copy()

# Group deliveries into overs
over_audit = (
    df_legal_audit
    .groupby(
        ["match_id", "innings", "overid"],
        as_index=False
    )
    .agg(
        Delivery_Count=("innings_ball_num", "size"),
        Max_Over_Ball_Num=("over_ball_num", "max"),
        Wide_Count=("extras1_runs", lambda x: (x == 2).sum()),
        Overstep_Count=("extras1_runs", lambda x: (x == 1).sum()),
        Other_Extras_Count=("extras1_runs", lambda x: (x == 5).sum()),
        Missing_Extras_Count=("extras_type", lambda x: x.isna().sum())
    )
)

# Identify overs containing more than six delivery records
extended_overs = over_audit[
    over_audit["Delivery_Count"] > 6
].copy()

# Identify extended overs without recorded wides or Oversteps
unexplained_extended_overs = extended_overs[
    (extended_overs["Wide_Count"] == 0) &
    (extended_overs["Overstep_Count"] == 0)
].copy()

print("TOTAL OVERS:", len(over_audit))

print("OVERS WITH MORE THAN SIX DELIVERY RECORDS:",
      len(extended_overs))

print("EXTENDED OVERS WITHOUT RECORDED WIDE/OVERSTEP:",
      len(unexplained_extended_overs))

print("\nEXTENDED OVERS — EXTRAS PROFILE")

display(
    extended_overs[
        [
            "Delivery_Count",
            "Max_Over_Ball_Num",
            "Wide_Count",
            "Overstep_Count",
            "Other_Extras_Count",
            "Missing_Extras_Count"
        ]
    ].value_counts().reset_index(name="Over_Count")
    .sort_values("Over_Count", ascending=False)
)

print("\nUNEXPLAINED EXTENDED OVERS")

display(
    unexplained_extended_overs.head(30)
)

TOTAL OVERS: 32350
OVERS WITH MORE THAN SIX DELIVERY RECORDS: 6710
EXTENDED OVERS WITHOUT RECORDED WIDE/OVERSTEP: 151

EXTENDED OVERS — EXTRAS PROFILE


,Delivery_Count,Max_Over_Ball_Num,Wide_Count,Overstep_Count,Other_Extras_Count,Missing_Extras_Count,Over_Count
0,7,7,1,0,0,6,4323
1,8,8,2,0,0,6,788
2,7,7,1,0,0,5,499
3,7,7,0,1,0,6,297
4,9,9,3,0,0,6,153
5,7,7,0,0,1,6,127
6,8,8,1,1,0,6,104
7,8,8,2,0,0,5,83
8,8,8,1,0,1,6,51
9,7,7,0,1,0,5,40



UNEXPLAINED EXTENDED OVERS


,match_id,innings,overid,Delivery_Count,Max_Over_Ball_Num,Wide_Count,Overstep_Count,Other_Extras_Count,Missing_Extras_Count
87,13513,1,10,7,7,0,0,1,6
196,13515,2,19,7,7,0,0,1,5
413,13614,1,18,7,7,0,0,1,6
417,13614,2,2,7,7,0,0,1,4
708,14098,2,20,7,7,0,0,1,6
804,14101,1,17,7,7,0,0,1,6
1055,14107,2,16,7,7,0,0,1,6
1089,14108,2,14,7,7,0,0,1,6
1091,14108,2,16,7,7,0,0,1,6
1172,14110,2,17,7,7,0,0,1,6


#### Inspect delivery-level sequences in unexplained overs

In [15]:
# Select the first 10 unexplained overs for detailed inspection
sample_overs = unexplained_extended_overs.head(10)

sample_keys = sample_overs[
    ["match_id", "innings", "overid"]
]

sample_deliveries = df_bb.merge(
    sample_keys,
    on=["match_id", "innings", "overid"],
    how="inner"
)

inspection_columns_legal = [
    "match_id",
    "innings",
    "overid",
    "over_ball_num",
    "innings_ball_num",
    "extras1_runs",
    "extras_type",
    "runs",
    "extras_2_runs",
    "is_wicket"
]

display(
    sample_deliveries[
        inspection_columns_legal
    ].sort_values(
        ["match_id", "innings", "overid", "over_ball_num"]
    ).reset_index(drop=True)
)

,match_id,innings,overid,over_ball_num,innings_ball_num,extras1_runs,extras_type,runs,extras_2_runs,is_wicket
0,13513,1,10,1,57,5,other,0,0,False
1,13513,1,10,2,58,6,NaN,6,0,False
2,13513,1,10,3,59,6,NaN,1,0,False
3,13513,1,10,4,60,6,NaN,2,0,False
4,13513,1,10,5,61,6,NaN,1,0,False
...,...,...,...,...,...,...,...,...,...,...
65,14110,2,17,3,100,6,NaN,1,0,False
66,14110,2,17,4,101,6,NaN,1,0,False
67,14110,2,17,5,102,6,NaN,4,0,False
68,14110,2,17,6,103,6,NaN,1,0,False


### Inspect match outcomes and innings structure

In [16]:
# Match outcome distribution
print("MATCH RESULT DISTRIBUTION")
display(
    df_matches_raw["result"]
    .value_counts(dropna=False)
    .rename_axis("result")
    .reset_index(name="match_count")
)

# Build a match-level innings and delivery summary
match_delivery_profile = (
    df_bb.groupby("match_id")
    .agg(
        delivery_records=("match_id", "size"),
        innings_count=("innings", "nunique"),
        innings_list=("innings", lambda x: sorted(x.dropna().unique().tolist()))
    )
    .reset_index()
)

match_population_audit = df_matches_raw[
    ["match_id", "comp_id", "result"]
].merge(
    match_delivery_profile,
    on="match_id",
    how="left"
)

match_population_audit["delivery_records"] = (
    match_population_audit["delivery_records"].fillna(0).astype(int)
)
match_population_audit["innings_count"] = (
    match_population_audit["innings_count"].fillna(0).astype(int)
)

print("MATCHES BY RESULT AND NUMBER OF RECORDED INNINGS")
display(
    match_population_audit
    .groupby(["result", "innings_count"], dropna=False)
    .size()
    .reset_index(name="match_count")
    .sort_values(["result", "innings_count"], na_position="last")
)

print("MATCHES WITH MORE THAN TWO RECORDED INNINGS")
display(
    match_population_audit[
        match_population_audit["innings_count"] > 2
    ].sort_values(["comp_id", "match_id"])
)

print("MATCHES WITH NO BALL-BY-BALL RECORDS")
display(
    match_population_audit[
        match_population_audit["delivery_records"] == 0
    ]
)

MATCH RESULT DISTRIBUTION


,result,match_count
0,No Result,21
1,Match Abandoned,19
2,India Won by 7 wickets,8
3,Delhi Capitals Won by 6 wickets,7
4,Kolkata Knight Riders Won by 8 wickets,6
...,...,...
665,South Africa Won by 6 wickets,1
666,Sri Lanka won the super over,1
667,New Zealand Won by 4 Wickets,1
668,Bangladesh Won by 6 Wickets,1


MATCHES BY RESULT AND NUMBER OF RECORDED INNINGS


,result,innings_count,match_count
0,Afghanistan A Won by 11 runs,2,1
1,Afghanistan A Won by 20 runs,2,1
2,Afghanistan A Won by 24 runs,2,1
3,Afghanistan A Won by 3 wickets,2,1
4,Afghanistan A Won by 4 wickets,2,1
...,...,...,...
667,Zimbabwe Won by 4 wickets,2,1
668,Zimbabwe Won by 5 Wickets,2,1
669,Zimbabwe Won by 5 wickets,2,1
670,Zimbabwe Won by 67 runs,2,1


MATCHES WITH MORE THAN TWO RECORDED INNINGS


,match_id,comp_id,result,delivery_records,innings_count,innings_list
478,34429,2542,India Tied with Srilanka (India Won the Super ...,257,4,"[1, 2, 3, 4]"
557,40725,2725,Delhi Capitals Tied with Rajasthan Royals (DC ...,262,4,"[1, 2, 3, 4]"
679,45512,3133,India won the Super Over(Match Tied),260,4,"[1, 2, 3, 4]"
745,48027,3289,Bangladesh A Won The Super Over,262,4,"[1, 2, 3, 4]"
747,48029,3289,Pakistan A Won the super over,257,4,"[1, 2, 3, 4]"
785,50359,3298,Kolkata Knight Riders Won by Super over (Match...,256,4,"[1, 2, 3, 4]"


MATCHES WITH NO BALL-BY-BALL RECORDS


,match_id,comp_id,result,delivery_records,innings_count,innings_list
396,31281,2193,Match Abandoned,0,0,NaN
399,31284,2193,Match Abandoned,0,0,NaN
403,31288,2193,Match Abandoned,0,0,NaN
425,32610,2354,Match Abandoned,0,0,NaN
427,32612,2354,Match Abandoned,0,0,NaN
447,32622,2398,Match Abandoned,0,0,NaN
453,38797,2442,Match Abandoned,0,0,NaN
481,36564,2548,Match Abandoned,0,0,NaN
498,36210,2628,Match Abandoned,0,0,NaN
593,40761,2725,Match Abandoned,0,0,NaN


In [17]:
# The six matches with four recorded innings
four_innings_ids = [34429, 40725, 45512, 48027, 48029, 50359]

# Match and competition metadata
print("COMPETITION TABLE SCHEMA")
print(df_competitions_raw.columns.tolist())
display(df_competitions_raw.head(10))

print("FOUR-INNINGS MATCH METADATA")
four_innings_matches = match_population_audit[
    match_population_audit["match_id"].isin(four_innings_ids)
].merge(
    df_matches_raw[
        [
            "match_id", "match_date", "stage", "venue_country",
            "home_team", "away_team", "result"
        ]
    ],
    on=["match_id", "result"],
    how="left"
)

display(
    four_innings_matches.sort_values("match_id")
)

# Delivery-level profile for each innings
four_innings_delivery_profile = (
    df_bb[df_bb["match_id"].isin(four_innings_ids)]
    .groupby(["match_id", "innings"], as_index=False)
    .agg(
        delivery_records=("match_id", "size"),
        overs_recorded=("overid", "nunique"),
        first_over=("overid", "min"),
        last_over=("overid", "max"),
        first_innings_ball=("innings_ball_num", "min"),
        last_innings_ball=("innings_ball_num", "max")
    )
    .sort_values(["match_id", "innings"])
)

print("DELIVERY PROFILE BY INNINGS")
display(four_innings_delivery_profile)

COMPETITION TABLE SCHEMA
['comp_id', 'comp_name', 'match_format', 'season']


,comp_id,comp_name,match_format,season
0,3395,South Africa in New Zealand T20I Series 2026,T20I,2026
1,3405,Afghanistan v Sri Lanka T20I Series 2026,T20I,2026
2,3465,India in Zimbabwe T20I Series 2026,T20I,2026
3,3568,Bangladesh in Zimbabwe T20I Series 2026,T20I,2026
4,3482,Australia in Bangladesh T20I Series 2026,T20I,2026
5,3464,Sri Lanka in West Indies T20I Series 2026,T20I,2026
6,3765,Sri Lanka in Pakistan T20I Series 2026,T20I,2026
7,3739,India in Japan T20I Match 2026,T20I,2026
8,3787,Afghanistan v Zimbabwe T20I Series 2026,T20I,2026
9,3713,Afghanistan vs India in India T20I Series 2026,T20I,2026


FOUR-INNINGS MATCH METADATA


,match_id,comp_id,result,delivery_records,innings_count,innings_list,match_date,stage,venue_country,home_team,away_team
0,34429,2542,India Tied with Srilanka (India Won the Super ...,257,4,"[1, 2, 3, 4]",2024-07-30,Others,Sri Lanka,Sri Lanka,India
1,40725,2725,Delhi Capitals Tied with Rajasthan Royals (DC ...,262,4,"[1, 2, 3, 4]",2025-04-16,League,India,Delhi Capitals,Rajasthan Royals
2,45512,3133,India won the Super Over(Match Tied),260,4,"[1, 2, 3, 4]",2025-09-26,Super 4,United Arab Emirates,India,Sri Lanka
3,48027,3289,Bangladesh A Won The Super Over,262,4,"[1, 2, 3, 4]",2025-11-21,Semi Final,Qatar,Bangladesh A,India A
4,48029,3289,Pakistan A Won the super over,257,4,"[1, 2, 3, 4]",2025-11-23,Final,Qatar,Bangladesh A,Pakistan A
5,50359,3298,Kolkata Knight Riders Won by Super over (Match...,256,4,"[1, 2, 3, 4]",2026-04-26,League,India,Lucknow Super Giants,Kolkata Knight Riders


DELIVERY PROFILE BY INNINGS


,match_id,innings,delivery_records,overs_recorded,first_over,last_over,first_innings_ball,last_innings_ball
0,34429,1,125,20,1,20,1,125
1,34429,2,127,20,1,20,1,127
2,34429,3,4,1,1,1,1,4
3,34429,4,1,1,1,1,1,1
4,40725,1,128,20,1,20,1,128
5,40725,2,124,20,1,20,1,124
6,40725,3,6,1,1,1,1,6
7,40725,4,4,1,1,1,1,4
8,45512,1,129,20,1,20,1,129
9,45512,2,124,20,1,20,1,124


### Re-construct the competition-format population

In [18]:
# Attach competition format to every match
df_match_scope_audit = df_matches_raw.merge(
    df_competitions_raw[["comp_id", "comp_name", "match_format", "season"]],
    on="comp_id",
    how="left",
    indicator=True
)

# Check whether every match maps to a competition
print("MATCHES WITHOUT A COMPETITION MATCH")
display(
    df_match_scope_audit[
        df_match_scope_audit["_merge"] != "both"
    ][["match_id", "comp_id", "_merge"]]
)

# Match counts by format
print("MATCH COUNTS BY COMPETITION FORMAT")
display(
    df_match_scope_audit
    .groupby("match_format", dropna=False)
    .agg(
        competition_count=("comp_id", "nunique"),
        match_count=("match_id", "nunique")
    )
    .reset_index()
    .sort_values("match_count", ascending=False)
)

# Attach innings counts to the competition audit
df_match_scope_audit = df_match_scope_audit.merge(
    match_delivery_profile[["match_id", "delivery_records", "innings_count", "innings_list"]],
    on="match_id",
    how="left"
)

# Check format and innings distribution
print("MATCH COUNTS BY FORMAT AND RECORDED INNINGS")
display(
    df_match_scope_audit
    .groupby(["match_format", "innings_count"], dropna=False)
    .size()
    .reset_index(name="match_count")
    .sort_values(["match_format", "innings_count"], na_position="last")
)

# Inspect the six four-innings matches with competition details
print("SIX FOUR-INNINGS MATCHES — COMPETITION DETAILS")
display(
    df_match_scope_audit[
        df_match_scope_audit["match_id"].isin(four_innings_ids)
    ][
        [
            "match_id", "comp_id", "comp_name", "match_format",
            "season", "match_date", "stage", "result",
            "delivery_records", "innings_count"
        ]
    ].sort_values("match_id")
)

MATCHES WITHOUT A COMPETITION MATCH


,match_id,comp_id,_merge


MATCH COUNTS BY COMPETITION FORMAT


,match_format,competition_count,match_count
1,T20I,131,482
0,T20,7,401


MATCH COUNTS BY FORMAT AND RECORDED INNINGS


,match_format,innings_count,match_count
0,T20,1.0,4
1,T20,2.0,389
2,T20,4.0,4
3,T20,NaN,4
4,T20I,1.0,18
5,T20I,2.0,450
6,T20I,4.0,2
7,T20I,NaN,12


SIX FOUR-INNINGS MATCHES — COMPETITION DETAILS


,match_id,comp_id,comp_name,match_format,season,match_date,stage,result,delivery_records,innings_count
478,34429,2542,India in Sri Lanka T20I Series 2024,T20I,2024,2024-07-30,Others,India Tied with Srilanka (India Won the Super ...,257.0,4.0
557,40725,2725,IPL 2025,T20,2025,2025-04-16,League,Delhi Capitals Tied with Rajasthan Royals (DC ...,262.0,4.0
679,45512,3133,Asia Cup 2025,T20I,2025,2025-09-26,Super 4,India won the Super Over(Match Tied),260.0,4.0
745,48027,3289,Asia Cup Rising Stars 2025,T20,2025,2025-11-21,Semi Final,Bangladesh A Won The Super Over,262.0,4.0
747,48029,3289,Asia Cup Rising Stars 2025,T20,2025,2025-11-23,Final,Pakistan A Won the super over,257.0,4.0
785,50359,3298,IPL 2026,T20,2026,2026-04-26,League,Kolkata Knight Riders Won by Super over (Match...,256.0,4.0


#### Compare match competitions with mapping coverage

In [27]:
# Competition IDs represented in the competition-team-player mapping
mapping_comp_ids = set(df_mapping_raw["comp_id"].dropna().unique())

# Competition IDs represented in the matches table
match_comp_ids = set(df_matches_raw["comp_id"].dropna().unique())

# Competition-level match and mapping coverage
competition_coverage = (
    df_match_scope_audit
    .groupby(["comp_id", "comp_name", "match_format"], dropna=False)
    .agg(
        match_count=("match_id", "nunique")
    )
    .reset_index()
)

mapping_coverage = (
    df_mapping_raw.groupby("comp_id")
    .agg(
        mapping_rows=("comp_id", "size"),
        mapped_teams=("team_id", "nunique"),
        mapped_players=("player_id", "nunique")
    )
    .reset_index()
)

competition_coverage = competition_coverage.merge(
    mapping_coverage,
    on="comp_id",
    how="left"
)

competition_coverage["has_mapping"] = (
    competition_coverage["mapping_rows"].notna()
)

print("COMPETITIONS WITHOUT MAPPING RECORDS")
display(
    competition_coverage[
        ~competition_coverage["has_mapping"]
    ].sort_values(["match_format", "comp_id"])
)

print("T20 COMPETITION COVERAGE")
display(
    competition_coverage[
        competition_coverage["match_format"] == "T20"
    ].sort_values("comp_id")
)

print("TOTAL MATCHES IN COMPETITIONS WITH MAPPING RECORDS")
display(
    competition_coverage.groupby(
        ["match_format", "has_mapping"], dropna=False
    )["match_count"].sum().reset_index()
)

COMPETITIONS WITHOUT MAPPING RECORDS


,comp_id,comp_name,match_format,match_count,mapping_rows,mapped_teams,mapped_players,has_mapping


T20 COMPETITION COVERAGE


,comp_id,comp_name,match_format,match_count,mapping_rows,mapped_teams,mapped_players,has_mapping
3,1358,IPL 2022,T20,74,251,10,251,True
31,1715,IPL 2023,T20,74,264,10,264,True
53,2193,IPL 2024,T20,74,263,10,263,True
79,2670,Mens T20 Emerging Teams Asia Cup 2024,T20,15,137,8,137,True
84,2725,IPL 2025,T20,75,255,10,255,True
117,3289,Asia Cup Rising Stars 2025,T20,15,140,8,140,True
118,3298,IPL 2026,T20,74,275,10,275,True


TOTAL MATCHES IN COMPETITIONS WITH MAPPING RECORDS


,match_format,has_mapping,match_count
0,T20,True,401
1,T20I,True,482


In [28]:
# Re-construct the df_target_matches population
# Target matches are T20 and T20I matches from competitions that have mapping records
target_comp_ids = competition_coverage[competition_coverage['has_mapping'] & competition_coverage['match_format'].isin(['T20', 'T20I'])]['comp_id'].unique()

df_target_matches = df_match_scope_audit[
    df_match_scope_audit['comp_id'].isin(target_comp_ids)
].copy()

print("Target matches base population reconstructed. Total matches:", len(df_target_matches))

Target matches base population reconstructed. Total matches: 883


### Build the target population and audit exclusions

In [30]:
# Inspect columns to confirm the duplicate merge
print("CURRENT TARGET MATCH COLUMNS:")
print(df_target_matches.columns.tolist())

# Remove any previously merged delivery-profile columns
profile_columns = [
    "delivery_records",
    "innings_count",
    "innings_list",
    "delivery_records_x",
    "delivery_records_y",
    "innings_count_x",
    "innings_count_y",
    "innings_list_x",
    "innings_list_y"
]

df_target_matches = df_target_matches.drop(
    columns=[
        col for col in profile_columns
        if col in df_target_matches.columns
    ]
)

# Merge the delivery profile exactly once
df_target_matches = df_target_matches.merge(
    match_delivery_profile[
        ["match_id", "delivery_records", "innings_count", "innings_list"]
    ],
    on="match_id",
    how="left",
    validate="one_to_one"
)

print("REPAIRED TARGET MATCH COLUMNS:")
print(df_target_matches.columns.tolist())

print("TARGET MATCHES BY RESULT AND INNINGS COUNT")
display(
    df_target_matches
    .groupby(["result", "innings_count"], dropna=False)
    .size()
    .reset_index(name="match_count")
    .sort_values(
        ["innings_count", "match_count"],
        ascending=[True, False]
    )
)

print("TARGET MATCHES WITH FEWER THAN TWO INNINGS")
display(
    df_target_matches[
        df_target_matches["innings_count"].fillna(0) < 2
    ][
        [
            "match_id", "comp_id", "comp_name", "match_format",
            "match_date", "result", "delivery_records",
            "innings_count", "innings_list"
        ]
    ].sort_values(["innings_count", "match_id"])
)

print("TARGET MATCHES WITH FOUR RECORDED INNINGS")
display(
    df_target_matches[
        df_target_matches["innings_count"] == 4
    ][
        [
            "match_id", "comp_id", "comp_name", "match_format",
            "match_date", "result", "delivery_records"
        ]
    ].sort_values("match_id")
)

CURRENT TARGET MATCH COLUMNS:
['match_id', 'comp_id', 'match_number', 'match_date', 'stage', 'venue_id', 'venue_country', 'home_team_id', 'home_team', 'away_team_id', 'away_team', 'home_team_captain_id', 'away_team_captain_id', 'toss_win_team_id', 'toss_opted', 'man_of_the_match', 'first_inn_revised_overs', 'second_inn_revised_overs', 'revised_target', 'knock_out', 'win_team_id', 'result', 'comp_name', 'match_format', 'season', '_merge', 'delivery_records_x', 'innings_count_x', 'innings_list_x', 'delivery_records_y', 'innings_count_y', 'innings_list_y']
REPAIRED TARGET MATCH COLUMNS:
['match_id', 'comp_id', 'match_number', 'match_date', 'stage', 'venue_id', 'venue_country', 'home_team_id', 'home_team', 'away_team_id', 'away_team', 'home_team_captain_id', 'away_team_captain_id', 'toss_win_team_id', 'toss_opted', 'man_of_the_match', 'first_inn_revised_overs', 'second_inn_revised_overs', 'revised_target', 'knock_out', 'win_team_id', 'result', 'comp_name', 'match_format', 'season', '_merge

,result,innings_count,match_count
397,No Result,1.0,19
330,Match Abandoned,1.0,3
238,India Won by 7 wickets,2.0,8
120,Delhi Capitals Won by 6 wickets,2.0,7
291,Kolkata Knight Riders Won by 8 wickets,2.0,6
...,...,...,...
104,Delhi Capitals Tied with Rajasthan Royals (DC ...,4.0,1
194,India Tied with Srilanka (India Won the Super ...,4.0,1
256,India won the Super Over(Match Tied),4.0,1
294,Kolkata Knight Riders Won by Super over (Match...,4.0,1


TARGET MATCHES WITH FEWER THAN TWO INNINGS


,match_id,comp_id,comp_name,match_format,match_date,result,delivery_records,innings_count,innings_list
94,15410,1439,South Africa in India T20I Series 2022,T20I,2022-06-19,Match Abandoned,22.0,1.0,[1]
95,15546,1452,Bangladesh in West Indies T20I Series 2022,T20I,2022-07-02,No Result,82.0,1.0,[1]
234,20226,1715,IPL 2023,T20,2023-05-03,Match Abandoned,118.0,1.0,[1]
281,21406,1799,New Zealand in Pakistan T20I Series 2023,T20I,2023-04-20,Match Abandoned,118.0,1.0,[1]
315,28333,2058,Bangladesh in New Zealand T20I Series 2023,T20I,2023-12-29,No Result,66.0,1.0,[1]
420,31707,2350,New Zealand in Pakistan T20I Series 2024,T20I,2024-04-18,No Result,2.0,1.0,[1]
488,35424,2605,Netherlands Tri-Nation T20I Series [Aug 2024] ...,T20I,2024-08-24,No Result,125.0,1.0,[1]
464,37353,2455,England in West Indies T20I Series 2024,T20I,2024-11-18,No Result,30.0,1.0,[1]
597,39983,2801,Ireland in Zimbabwe T20I Series 2025,T20I,2025-02-22,No Result,56.0,1.0,[1]
599,39985,2801,Ireland in Zimbabwe T20I Series 2025,T20I,2025-02-25,No Result,113.0,1.0,[1]


TARGET MATCHES WITH FOUR RECORDED INNINGS


,match_id,comp_id,comp_name,match_format,match_date,result,delivery_records
478,34429,2542,India in Sri Lanka T20I Series 2024,T20I,2024-07-30,India Tied with Srilanka (India Won the Super ...,257.0
542,40725,2725,IPL 2025,T20,2025-04-16,Delhi Capitals Tied with Rajasthan Royals (DC ...,262.0
664,45512,3133,Asia Cup 2025,T20I,2025-09-26,India won the Super Over(Match Tied),260.0
755,50359,3298,IPL 2026,T20,2026-04-26,Kolkata Knight Riders Won by Super over (Match...,256.0


### Inspect the two-innings No Result matches

In [31]:
# Identify the two No Result matches with two recorded innings
two_innings_no_result = df_target_matches[
    (df_target_matches["result"] == "No Result") &
    (df_target_matches["innings_count"] == 2)
].copy()

print("TWO-INNINGS NO RESULT MATCHES")
display(
    two_innings_no_result[
        [
            "match_id", "comp_id", "comp_name", "match_format",
            "match_date", "result", "delivery_records",
            "first_inn_revised_overs", "second_inn_revised_overs"
        ]
    ].sort_values("match_id")
)

# Inspect delivery structure within each innings
no_result_ids = two_innings_no_result["match_id"].tolist()

no_result_innings_profile = (
    df_bb[df_bb["match_id"].isin(no_result_ids)]
    .groupby(["match_id", "innings"], as_index=False)
    .agg(
        delivery_records=("match_id", "size"),
        overs_recorded=("overid", "nunique"),
        first_over=("overid", "min"),
        last_over=("overid", "max"),
        max_over_ball_num=("over_ball_num", "max"),
        max_innings_ball_num=("innings_ball_num", "max"),
        wickets=("is_wicket", "sum")
    )
    .sort_values(["match_id", "innings"])
)

print("INNINGS-LEVEL DELIVERY PROFILE")
display(no_result_innings_profile)

TWO-INNINGS NO RESULT MATCHES


,match_id,comp_id,comp_name,match_format,match_date,result,delivery_records,first_inn_revised_overs,second_inn_revised_overs
177,17765,1611,England in Australia T20I Series 2022,T20I,2022-10-14,No Result,100.0,72.0,72.0
554,40737,2725,IPL 2025,T20,2025-04-26,No Result,130.0,NaN,NaN


INNINGS-LEVEL DELIVERY PROFILE


,match_id,innings,delivery_records,overs_recorded,first_over,last_over,max_over_ball_num,max_innings_ball_num,wickets
0,17765,1,75,12,1,12,7,75,2
1,17765,2,25,4,1,4,7,25,3
2,40737,1,123,20,1,20,7,123,4
3,40737,2,7,1,1,1,7,7,0


### Create the approved eligible population

In [32]:
# Phase 1 exclusion categories
excluded_results = ["No Result", "Match Abandoned"]

# Eligible completed-match population
df_eligible_matches = df_target_matches[
    ~df_target_matches["result"].isin(excluded_results)
].copy()

eligible_match_ids = set(df_eligible_matches["match_id"])

# Split eligible deliveries into regulation and Super Over subsets
df_eligible_deliveries = df_bb[
    df_bb["match_id"].isin(eligible_match_ids)
].copy()

df_eligible_regulation_deliveries = df_eligible_deliveries[
    df_eligible_deliveries["innings"].isin([1, 2])
].copy()

df_eligible_super_over_deliveries = df_eligible_deliveries[
    df_eligible_deliveries["innings"].isin([3, 4])
].copy()

# Population reconciliation
print("ELIGIBLE MATCH COUNTS")
display(
    df_eligible_matches.groupby("match_format")
    .agg(match_count=("match_id", "nunique"))
    .reset_index()
)

print("TOTAL ELIGIBLE MATCHES:", df_eligible_matches["match_id"].nunique())
print("EXCLUDED TARGET MATCHES:", len(df_target_matches) - len(df_eligible_matches))

print("\nDELIVERY POPULATION")
print("Eligible match deliveries:", len(df_eligible_deliveries))
print("Regulation deliveries (innings 1 and 2):", len(df_eligible_regulation_deliveries))
print("Super Over deliveries (innings 3 and 4):", len(df_eligible_super_over_deliveries))

print("\nINNINGS DISTRIBUTION — ELIGIBLE DELIVERIES")
display(
    df_eligible_deliveries["innings"]
    .value_counts(dropna=False)
    .sort_index()
    .rename_axis("innings")
    .reset_index(name="delivery_count")
)

# Assertions against Phase 1 approved counts
assert df_eligible_matches["match_id"].nunique() == 813
assert df_eligible_matches["match_format"].value_counts().to_dict() == {
    "T20I": 451,
    "T20": 362
}
assert len(df_eligible_regulation_deliveries) == 192235
assert len(df_eligible_super_over_deliveries) == 26

print("\nPASS: All approved Phase 1 population counts reconciled.")

ELIGIBLE MATCH COUNTS


,match_format,match_count
0,T20,362
1,T20I,451


TOTAL ELIGIBLE MATCHES: 813
EXCLUDED TARGET MATCHES: 40

DELIVERY POPULATION
Eligible match deliveries: 192261
Regulation deliveries (innings 1 and 2): 192235
Super Over deliveries (innings 3 and 4): 26

INNINGS DISTRIBUTION — ELIGIBLE DELIVERIES


,innings,delivery_count
0,1,100333
1,2,91902
2,3,19
3,4,7



PASS: All approved Phase 1 population counts reconciled.


### Re-run the extended-over audit on regulation deliveries only

In [33]:
# Legal-ball audit: approved regulation population only
df_legal_audit = df_eligible_regulation_deliveries.copy()

over_audit = (
    df_legal_audit
    .groupby(["match_id", "innings", "overid"], as_index=False)
    .agg(
        Delivery_Count=("innings_ball_num", "size"),
        Max_Over_Ball_Num=("over_ball_num", "max"),
        Wide_Count=("extras1_runs", lambda x: (x == 2).sum()),
        Overstep_Count=("extras1_runs", lambda x: (x == 1).sum()),
        Other_Extras_Count=("extras1_runs", lambda x: (x == 5).sum()),
        Missing_Extras_Count=("extras_type", lambda x: x.isna().sum())
    )
)

extended_overs = over_audit[
    over_audit["Delivery_Count"] > 6
].copy()

unexplained_extended_overs = extended_overs[
    (extended_overs["Wide_Count"] == 0) &
    (extended_overs["Overstep_Count"] == 0)
].copy()

print("TOTAL REGULATION OVERS:", len(over_audit))
print("OVERS WITH MORE THAN SIX DELIVERY RECORDS:", len(extended_overs))
print(
    "EXTENDED OVERS WITHOUT RECORDED WIDE/OVERSTEP:",
    len(unexplained_extended_overs)
)

print("\nEXTENDED OVERS — DELIVERY COUNT DISTRIBUTION")
display(
    extended_overs["Delivery_Count"]
    .value_counts()
    .sort_index()
    .rename_axis("Delivery_Count")
    .reset_index(name="Over_Count")
)

print("\nUNEXPLAINED EXTENDED OVERS — EXTRAS PROFILE")
display(
    unexplained_extended_overs[
        [
            "Delivery_Count", "Max_Over_Ball_Num",
            "Wide_Count", "Overstep_Count",
            "Other_Extras_Count", "Missing_Extras_Count"
        ]
    ]
    .value_counts()
    .reset_index(name="Over_Count")
    .sort_values(
        ["Delivery_Count", "Other_Extras_Count"],
        ascending=[True, True]
    )
)

# Compare with Phase 1 audit counts
print("\nPHASE 1 RECONCILIATION")
print("Expected extended overs: 6450")
print("Observed extended overs:", len(extended_overs))
print("Expected unexplained extended overs: 147")
print("Observed unexplained extended overs:", len(unexplained_extended_overs))

TOTAL REGULATION OVERS: 30921
OVERS WITH MORE THAN SIX DELIVERY RECORDS: 6450
EXTENDED OVERS WITHOUT RECORDED WIDE/OVERSTEP: 147

EXTENDED OVERS — DELIVERY COUNT DISTRIBUTION


,Delivery_Count,Over_Count
0,7,5128
1,8,1048
2,9,218
3,10,41
4,11,13
5,13,2



UNEXPLAINED EXTENDED OVERS — EXTRAS PROFILE


,Delivery_Count,Max_Over_Ball_Num,Wide_Count,Overstep_Count,Other_Extras_Count,Missing_Extras_Count,Over_Count
4,7,7,0,0,0,7,1
0,7,7,0,0,1,6,124
1,7,7,0,0,1,5,13
3,7,7,0,0,1,4,3
2,8,8,0,0,2,6,4
5,8,8,0,0,2,4,1
6,9,9,0,0,3,6,1



PHASE 1 RECONCILIATION
Expected extended overs: 6450
Observed extended overs: 6450
Expected unexplained extended overs: 147
Observed unexplained extended overs: 147


### Inspect the 147 unexplained overs in detail

In [34]:
# Collect keys for all 147 unexplained extended overs
unexplained_keys = unexplained_extended_overs[
    ["match_id", "innings", "overid"]
].copy()

# Retrieve every delivery in those overs
unexplained_deliveries = df_legal_audit.merge(
    unexplained_keys,
    on=["match_id", "innings", "overid"],
    how="inner"
)

inspection_columns = [
    "match_id",
    "innings",
    "overid",
    "over_ball_num",
    "innings_ball_num",
    "striker_id",
    "non_striker_id",
    "bowler_id",
    "extras1_runs",
    "extras_type",
    "runs",
    "extras_2_runs",
    "is_wicket",
    "dismissal_type"
]

print("TOTAL DELIVERY RECORDS IN 147 OVERS:", len(unexplained_deliveries))

print("\nEXTRAS CATEGORY DISTRIBUTION")
display(
    unexplained_deliveries["extras_type"]
    .value_counts(dropna=False)
    .rename_axis("extras_type")
    .reset_index(name="delivery_count")
)

print("\nEXTRAS1 CODE DISTRIBUTION")
display(
    unexplained_deliveries["extras1_runs"]
    .value_counts(dropna=False)
    .sort_index()
    .rename_axis("extras1_runs")
    .reset_index(name="delivery_count")
)

print("\nDELIVERY-LEVEL RECORDS — FIRST 50 ROWS")
display(
    unexplained_deliveries[
        inspection_columns
    ]
    .sort_values(["match_id", "innings", "overid", "over_ball_num"])
    .head(50)
)

TOTAL DELIVERY RECORDS IN 147 OVERS: 1036

EXTRAS CATEGORY DISTRIBUTION


,extras_type,delivery_count
0,NaN,862
1,other,153
2,legbyes,14
3,byes,7



EXTRAS1 CODE DISTRIBUTION


,extras1_runs,delivery_count
0,3,14
1,4,7
2,5,153
3,6,862



DELIVERY-LEVEL RECORDS — FIRST 50 ROWS


,match_id,innings,overid,over_ball_num,innings_ball_num,striker_id,non_striker_id,bowler_id,extras1_runs,extras_type,runs,extras_2_runs,is_wicket,dismissal_type
0,13513,1,10,1,57,4986,3125,2304,5,other,0,0,False,NaN
1,13513,1,10,2,58,4986,3125,2304,6,NaN,6,0,False,NaN
2,13513,1,10,3,59,4986,3125,2304,6,NaN,1,0,False,NaN
3,13513,1,10,4,60,3125,4986,2304,6,NaN,2,0,False,NaN
4,13513,1,10,5,61,3125,4986,2304,6,NaN,1,0,False,NaN
5,13513,1,10,6,62,4986,3125,2304,6,NaN,0,0,False,NaN
6,13513,1,10,7,63,4986,3125,2304,6,NaN,1,0,False,NaN
7,13515,2,19,1,115,1924,786,1772,6,NaN,1,0,False,NaN
8,13515,2,19,2,116,786,1924,1772,6,NaN,0,0,False,NaN
9,13515,2,19,3,117,786,1924,1772,6,NaN,1,0,False,NaN


#### Quantify unresolved extras by over

In [35]:
# Aggregate unresolved extras within each of the 147 overs
unresolved_over_profile = (
    unexplained_deliveries
    .groupby(["match_id", "innings", "overid"], as_index=False)
    .agg(
        Delivery_Count=("match_id", "size"),
        Other_Count=("extras1_runs", lambda x: (x == 5).sum()),
        Bye_Count=("extras1_runs", lambda x: (x == 4).sum()),
        Legbye_Count=("extras1_runs", lambda x: (x == 3).sum()),
        Missing_Extras_Count=("extras_type", lambda x: x.isna().sum()),
        Extras2_Nonzero_Count=("extras_2_runs", lambda x: (x != 0).sum()),
        Extras2_Total=("extras_2_runs", "sum"),
        Runs_Total=("runs", "sum")
    )
)

print("UNRESOLVED OVER PROFILE — SUMMARY")
display(
    unresolved_over_profile[
        [
            "Other_Count", "Bye_Count", "Legbye_Count",
            "Missing_Extras_Count", "Extras2_Nonzero_Count"
        ]
    ]
    .value_counts()
    .reset_index(name="Over_Count")
    .sort_values("Over_Count", ascending=False)
)

print("OVERS WITH NONZERO EXTRAS_2_RUNS")
display(
    unresolved_over_profile[
        unresolved_over_profile["Extras2_Nonzero_Count"] > 0
    ].sort_values(["match_id", "innings", "overid"])
)

print("TOTALS ACROSS THE 147 OVERS")
display(
    unresolved_over_profile[
        [
            "Delivery_Count", "Other_Count", "Bye_Count",
            "Legbye_Count", "Missing_Extras_Count",
            "Extras2_Nonzero_Count", "Extras2_Total", "Runs_Total"
        ]
    ].sum()
    .rename("total")
    .to_frame()
)

UNRESOLVED OVER PROFILE — SUMMARY


,Other_Count,Bye_Count,Legbye_Count,Missing_Extras_Count,Extras2_Nonzero_Count,Over_Count
0,1,0,0,6,0,116
1,1,0,0,6,1,8
2,1,0,1,5,0,8
3,1,1,0,5,0,4
4,2,0,0,6,0,4
5,1,0,2,4,0,3
6,0,0,0,7,0,1
7,1,1,0,5,1,1
8,2,2,0,4,0,1
9,3,0,0,6,0,1


OVERS WITH NONZERO EXTRAS_2_RUNS


,match_id,innings,overid,Delivery_Count,Other_Count,Bye_Count,Legbye_Count,Missing_Extras_Count,Extras2_Nonzero_Count,Extras2_Total,Runs_Total
2,13614,1,18,7,1,0,0,6,1,4,8
40,20234,2,13,7,1,0,0,6,1,4,12
44,20243,2,19,7,1,0,0,6,1,4,14
53,21583,1,20,7,1,0,0,6,1,1,13
90,37278,1,12,7,1,0,0,6,1,4,8
93,40038,1,19,7,1,0,0,6,1,1,9
94,40695,1,18,7,1,0,0,6,1,4,18
123,48467,1,20,7,1,0,0,6,1,1,12
146,55078,1,20,7,1,1,0,5,1,1,7


TOTALS ACROSS THE 147 OVERS


,total
Delivery_Count,1036
Other_Count,153
Bye_Count,7
Legbye_Count,14
Missing_Extras_Count,862
Extras2_Nonzero_Count,9
Extras2_Total,24
Runs_Total,1799


#### Create a provisional delivery classification

In [36]:
# Work on a separate copy; preserve the original regulation data
df_delivery_classified = df_eligible_regulation_deliveries.copy()

# Create explicit source-based event categories
df_delivery_classified["delivery_event_class"] = "UNRESOLVED_OR_OTHER"

df_delivery_classified.loc[
    df_delivery_classified["extras1_runs"] == 2,
    "delivery_event_class"
] = "RECORDED_WIDE"

df_delivery_classified.loc[
    df_delivery_classified["extras1_runs"] == 1,
    "delivery_event_class"
] = "RECORDED_OVERSTEP"

# Flag deliveries belonging to the 147 unexplained extended overs
unexplained_over_keys = set(
    zip(
        unexplained_extended_overs["match_id"],
        unexplained_extended_overs["innings"],
        unexplained_extended_overs["overid"]
    )
)

df_delivery_classified["unexplained_extended_over"] = [
    (match_id, innings, overid) in unexplained_over_keys
    for match_id, innings, overid in zip(
        df_delivery_classified["match_id"],
        df_delivery_classified["innings"],
        df_delivery_classified["overid"]
    )
]

# Summary of provisional classification
print("PROVISIONAL DELIVERY EVENT CLASSIFICATION")
display(
    df_delivery_classified["delivery_event_class"]
    .value_counts(dropna=False)
    .rename_axis("delivery_event_class")
    .reset_index(name="delivery_count")
)

print("DELIVERIES IN UNEXPLAINED EXTENDED OVERS")
display(
    df_delivery_classified["unexplained_extended_over"]
    .value_counts()
    .rename_axis("unexplained_extended_over")
    .reset_index(name="delivery_count")
)

print("CLASSIFICATION CROSS-TAB")
display(
    pd.crosstab(
        df_delivery_classified["delivery_event_class"],
        df_delivery_classified["unexplained_extended_over"],
        margins=True
    )
)

# Confirm source data was not modified
assert len(df_delivery_classified) == len(df_eligible_regulation_deliveries)
assert df_delivery_classified["delivery_event_class"].notna().all()

print("PASS: Provisional labels created; source event fields preserved.")

PROVISIONAL DELIVERY EVENT CLASSIFICATION


,delivery_event_class,delivery_count
0,UNRESOLVED_OR_OTHER,184259
1,RECORDED_WIDE,7419
2,RECORDED_OVERSTEP,557


DELIVERIES IN UNEXPLAINED EXTENDED OVERS


,unexplained_extended_over,delivery_count
0,False,191199
1,True,1036


CLASSIFICATION CROSS-TAB


unexplained_extended_over,False,True,All
delivery_event_class,,,
RECORDED_OVERSTEP,557,0,557
RECORDED_WIDE,7419,0,7419
UNRESOLVED_OR_OTHER,183223,1036,184259
All,191199,1036,192235


PASS: Provisional labels created; source event fields preserved.


#### Refine source event labels

In [37]:
# Create a new copy; preserve the existing provisional classification and raw data.
df_delivery_classified_v2 = df_eligible_regulation_deliveries.copy()

# Map the source extras code without inferring legal/illegal status.
extras_code_map = {
    1: "RECORDED_OVERSTEP",
    2: "RECORDED_WIDE",
    3: "RECORDED_LEGBYES",
    4: "RECORDED_BYES",
    5: "RECORDED_OTHER",
    6: "RECORDED_NO_EXTRAS",
    7: "RECORDED_PENALTY"
}

df_delivery_classified_v2["delivery_event_class"] = (
    df_delivery_classified_v2["extras1_runs"]
    .map(extras_code_map)
    .fillna("UNMAPPED_EXTRAS_CODE")
)

# Keep missing extras_type as a separate observable condition.
df_delivery_classified_v2["extras_type_missing"] = (
    df_delivery_classified_v2["extras_type"].isna()
)

# Preserve the unexplained extended-over flag independently.
df_delivery_classified_v2["unexplained_extended_over"] = (
    df_delivery_classified_v2["unexplained_extended_over"]
    if "unexplained_extended_over" in df_delivery_classified_v2.columns
    else [
        (match_id, innings, overid) in unexplained_over_keys
        for match_id, innings, overid in zip(
            df_delivery_classified_v2["match_id"],
            df_delivery_classified_v2["innings"],
            df_delivery_classified_v2["overid"]
        )
    ]
)

# Summarise event classes.
print("SOURCE-CODE EVENT CLASSIFICATION")
display(
    df_delivery_classified_v2["delivery_event_class"]
    .value_counts(dropna=False)
    .rename_axis("delivery_event_class")
    .reset_index(name="delivery_count")
)

print("CLASSIFICATION BY UNEXPLAINED EXTENDED OVER")
display(
    pd.crosstab(
        df_delivery_classified_v2["delivery_event_class"],
        df_delivery_classified_v2["unexplained_extended_over"],
        margins=True
    )
)

print("MISSING EXTRAS TYPE BY EVENT CLASS")
display(
    pd.crosstab(
        df_delivery_classified_v2["delivery_event_class"],
        df_delivery_classified_v2["extras_type_missing"],
        margins=True
    )
)

assert len(df_delivery_classified_v2) == 192235
assert df_delivery_classified_v2["delivery_event_class"].notna().all()
assert df_delivery_classified_v2["unexplained_extended_over"].sum() == 1036

print("PASS: Source-code labels assigned; no legal-ball inference made.")

SOURCE-CODE EVENT CLASSIFICATION


,delivery_event_class,delivery_count
0,RECORDED_NO_EXTRAS,180862
1,RECORDED_WIDE,7419
2,RECORDED_LEGBYES,2635
3,RECORDED_OVERSTEP,557
4,RECORDED_BYES,505
5,RECORDED_OTHER,256
6,RECORDED_PENALTY,1


CLASSIFICATION BY UNEXPLAINED EXTENDED OVER


unexplained_extended_over,False,True,All
delivery_event_class,,,
RECORDED_BYES,498,7,505
RECORDED_LEGBYES,2621,14,2635
RECORDED_NO_EXTRAS,180000,862,180862
RECORDED_OTHER,103,153,256
RECORDED_OVERSTEP,557,0,557
RECORDED_PENALTY,1,0,1
RECORDED_WIDE,7419,0,7419
All,191199,1036,192235


MISSING EXTRAS TYPE BY EVENT CLASS


extras_type_missing,False,True,All
delivery_event_class,,,
RECORDED_BYES,505,0,505
RECORDED_LEGBYES,2635,0,2635
RECORDED_NO_EXTRAS,0,180862,180862
RECORDED_OTHER,256,0,256
RECORDED_OVERSTEP,557,0,557
RECORDED_PENALTY,1,0,1
RECORDED_WIDE,7419,0,7419
All,11373,180862,192235


PASS: Source-code labels assigned; no legal-ball inference made.


#### Quantify unresolved delivery evidence

In [38]:
# Create a compact evidence profile for the 1,036 deliveries
# in unexplained extended overs.

df_unexplained_delivery_evidence = df_delivery_classified_v2[
    df_delivery_classified_v2["unexplained_extended_over"]
].copy()

df_unexplained_delivery_evidence["extras_2_nonzero"] = (
    df_unexplained_delivery_evidence["extras_2_runs"] != 0
)

evidence_profile = (
    df_unexplained_delivery_evidence
    .groupby(
        ["delivery_event_class", "extras_2_nonzero"],
        dropna=False
    )
    .size()
    .reset_index(name="delivery_count")
    .sort_values(
        ["delivery_event_class", "extras_2_nonzero"]
    )
)

print("UNEXPLAINED EXTENDED-OVER DELIVERY EVIDENCE")
display(evidence_profile)

print("EXTRAS_2_RUNS DISTRIBUTION")
display(
    df_unexplained_delivery_evidence["extras_2_runs"]
    .value_counts(dropna=False)
    .sort_index()
    .rename_axis("extras_2_runs")
    .reset_index(name="delivery_count")
)

print("UNEXPLAINED OVER SUMMARY")
display(
    df_unexplained_delivery_evidence.groupby(
        ["match_id", "innings", "overid"],
        as_index=False
    ).agg(
        delivery_count=("match_id", "size"),
        event_classes=("delivery_event_class", "nunique"),
        extras_2_nonzero_deliveries=("extras_2_nonzero", "sum"),
        extras_2_total=("extras_2_runs", "sum"),
        runs_total=("runs", "sum")
    ).head(20)
)

assert len(df_unexplained_delivery_evidence) == 1036
assert df_unexplained_delivery_evidence["extras_2_nonzero"].sum() == 9

print("PASS: Unexplained-over evidence profile generated.")

UNEXPLAINED EXTENDED-OVER DELIVERY EVIDENCE


,delivery_event_class,extras_2_nonzero,delivery_count
0,RECORDED_BYES,False,7
1,RECORDED_LEGBYES,False,14
2,RECORDED_NO_EXTRAS,False,862
3,RECORDED_OTHER,False,144
4,RECORDED_OTHER,True,9


EXTRAS_2_RUNS DISTRIBUTION


,extras_2_runs,delivery_count
0,0,1027
1,1,4
2,4,5


UNEXPLAINED OVER SUMMARY


,match_id,innings,overid,delivery_count,event_classes,extras_2_nonzero_deliveries,extras_2_total,runs_total
0,13513,1,10,7,2,0,0,11
1,13515,2,19,7,3,0,0,5
2,13614,1,18,7,2,1,4,8
3,13614,2,2,7,3,0,0,5
4,14098,2,20,7,2,0,0,14
5,14101,1,17,7,2,0,0,15
6,14107,2,16,7,2,0,0,34
7,14108,2,14,7,2,0,0,13
8,14108,2,16,7,2,0,0,10
9,14110,2,17,7,2,0,0,18


PASS: Unexplained-over evidence profile generated.


### Inspect all 147 overs and the 9 ambiguous records

In [39]:
# Complete over-level evidence table: all 147 unexplained extended overs.

df_unexplained_over_full = (
    df_unexplained_delivery_evidence
    .groupby(["match_id", "innings", "overid"], as_index=False)
    .agg(
        delivery_count=("match_id", "size"),
        event_classes=("delivery_event_class", "nunique"),
        recorded_other=("delivery_event_class",
                        lambda x: (x == "RECORDED_OTHER").sum()),
        recorded_byes=("delivery_event_class",
                       lambda x: (x == "RECORDED_BYES").sum()),
        recorded_legbyes=("delivery_event_class",
                          lambda x: (x == "RECORDED_LEGBYES").sum()),
        recorded_no_extras=("delivery_event_class",
                            lambda x: (x == "RECORDED_NO_EXTRAS").sum()),
        extras_2_nonzero_deliveries=("extras_2_nonzero", "sum"),
        extras_2_total=("extras_2_runs", "sum"),
        runs_total=("runs", "sum")
    )
    .sort_values(["match_id", "innings", "overid"])
    .reset_index(drop=True)
)

print("COMPLETE UNEXPLAINED OVER REGISTER")
print("Rows:", len(df_unexplained_over_full))
display(df_unexplained_over_full)

# Inspect every delivery with nonzero extras_2_runs.
df_extras2_nonzero_detail = (
    df_unexplained_delivery_evidence[
        df_unexplained_delivery_evidence["extras_2_runs"] != 0
    ].copy()
)

preferred_columns = [
    "match_id", "innings", "overid",
    "innings_ball_num", "over_ball_num",
    "delivery_event_class", "extras1_runs",
    "extras_type", "extras_2_runs", "runs",
    "batsman_id", "bowler_id", "dismissal_type",
    "dismissed_player_id"
]

detail_columns = [
    col for col in preferred_columns
    if col in df_extras2_nonzero_detail.columns
]

print("ALL NONZERO EXTRAS_2_RUNS DELIVERY RECORDS")
display(
    df_extras2_nonzero_detail[detail_columns]
    .sort_values(["match_id", "innings", "overid", "innings_ball_num"])
)

assert len(df_unexplained_over_full) == 147
assert len(df_extras2_nonzero_detail) == 9
assert df_unexplained_over_full["delivery_count"].sum() == 1036
assert df_unexplained_over_full["extras_2_total"].sum() == 24

print("PASS: Complete 147-over register and all 9 nonzero extras_2_runs records extracted.")

COMPLETE UNEXPLAINED OVER REGISTER
Rows: 147


,match_id,innings,overid,delivery_count,event_classes,recorded_other,recorded_byes,recorded_legbyes,recorded_no_extras,extras_2_nonzero_deliveries,extras_2_total,runs_total
0,13513,1,10,7,2,1,0,0,6,0,0,11
1,13515,2,19,7,3,1,0,1,5,0,0,5
2,13614,1,18,7,2,1,0,0,6,1,4,8
3,13614,2,2,7,3,1,0,2,4,0,0,5
4,14098,2,20,7,2,1,0,0,6,0,0,14
...,...,...,...,...,...,...,...,...,...,...,...,...
142,50943,2,19,7,2,1,0,0,6,0,0,3
143,54210,1,16,7,2,1,0,0,6,0,0,13
144,54210,2,16,7,2,1,0,0,6,0,0,6
145,54213,1,16,7,2,1,0,0,6,0,0,31


ALL NONZERO EXTRAS_2_RUNS DELIVERY RECORDS


,match_id,innings,overid,innings_ball_num,over_ball_num,delivery_event_class,extras1_runs,extras_type,extras_2_runs,runs,bowler_id,dismissal_type,dismissed_player_id
2588,13614,1,18,111,5,RECORDED_OTHER,5,other,4,0,829,NaN,NaN
59011,20234,2,13,80,3,RECORDED_OTHER,5,other,4,0,2261,NaN,NaN
61163,20243,2,19,121,4,RECORDED_OTHER,5,other,4,0,5809,NaN,NaN
66138,21583,1,20,119,4,RECORDED_OTHER,5,other,1,0,1566,NaN,NaN
114427,37278,1,12,69,1,RECORDED_OTHER,5,other,4,0,2681,NaN,NaN
123548,40038,1,19,114,3,RECORDED_OTHER,5,other,1,0,1566,NaN,NaN
124457,40695,1,18,115,1,RECORDED_OTHER,5,other,4,0,3298,NaN,NaN
170542,48467,1,20,126,6,RECORDED_OTHER,5,other,1,0,42548,NaN,NaN
199755,55078,1,20,124,4,RECORDED_OTHER,5,other,1,0,71727,NaN,NaN


PASS: Complete 147-over register and all 9 nonzero extras_2_runs records extracted.


#### Create a documented exception register

In [40]:
# Extract the nine records with nonzero extras_2_runs.
df_extras2_exceptions = df_extras2_nonzero_detail.copy()

df_extras2_exceptions["exception_id"] = [
    f"P2-EXTRAS2-{i:03d}"
    for i in range(1, len(df_extras2_exceptions) + 1)
]

df_extras2_exceptions["exception_type"] = (
    "NONZERO_EXTRAS2_WITH_RECORDED_OTHER"
)

df_extras2_exceptions["treatment"] = (
    "PRESERVE_SOURCE_VALUE; SEMANTICS_UNRESOLVED"
)

# Keep only useful audit fields.
exception_columns = [
    "exception_id",
    "exception_type",
    "match_id",
    "innings",
    "overid",
    "innings_ball_num",
    "bowler_id",
    "extras1_runs",
    "extras_type",
    "extras_2_runs",
    "runs",
    "treatment"
]

df_extras2_exceptions = df_extras2_exceptions[
    exception_columns
].sort_values(
    ["match_id", "innings", "overid"]
).reset_index(drop=True)

display(df_extras2_exceptions)

# Append to the existing exception register without changing prior entries.
phase2_exceptions = pd.concat(
    [
        phase2_exceptions,
        df_extras2_exceptions
    ],
    ignore_index=True
)

print("New extras_2 exceptions:", len(df_extras2_exceptions))
print("Total Phase 2 exceptions:", len(phase2_exceptions))

assert len(df_extras2_exceptions) == 9
assert (
    df_extras2_exceptions["treatment"]
    == "PRESERVE_SOURCE_VALUE; SEMANTICS_UNRESOLVED"
).all()

print("PASS: Nine extras_2 exceptions documented; source values preserved.")

,exception_id,exception_type,match_id,innings,overid,innings_ball_num,bowler_id,extras1_runs,extras_type,extras_2_runs,runs,treatment
0,P2-EXTRAS2-001,NONZERO_EXTRAS2_WITH_RECORDED_OTHER,13614,1,18,111,829,5,other,4,0,PRESERVE_SOURCE_VALUE; SEMANTICS_UNRESOLVED
1,P2-EXTRAS2-002,NONZERO_EXTRAS2_WITH_RECORDED_OTHER,20234,2,13,80,2261,5,other,4,0,PRESERVE_SOURCE_VALUE; SEMANTICS_UNRESOLVED
2,P2-EXTRAS2-003,NONZERO_EXTRAS2_WITH_RECORDED_OTHER,20243,2,19,121,5809,5,other,4,0,PRESERVE_SOURCE_VALUE; SEMANTICS_UNRESOLVED
3,P2-EXTRAS2-004,NONZERO_EXTRAS2_WITH_RECORDED_OTHER,21583,1,20,119,1566,5,other,1,0,PRESERVE_SOURCE_VALUE; SEMANTICS_UNRESOLVED
4,P2-EXTRAS2-005,NONZERO_EXTRAS2_WITH_RECORDED_OTHER,37278,1,12,69,2681,5,other,4,0,PRESERVE_SOURCE_VALUE; SEMANTICS_UNRESOLVED
5,P2-EXTRAS2-006,NONZERO_EXTRAS2_WITH_RECORDED_OTHER,40038,1,19,114,1566,5,other,1,0,PRESERVE_SOURCE_VALUE; SEMANTICS_UNRESOLVED
6,P2-EXTRAS2-007,NONZERO_EXTRAS2_WITH_RECORDED_OTHER,40695,1,18,115,3298,5,other,4,0,PRESERVE_SOURCE_VALUE; SEMANTICS_UNRESOLVED
7,P2-EXTRAS2-008,NONZERO_EXTRAS2_WITH_RECORDED_OTHER,48467,1,20,126,42548,5,other,1,0,PRESERVE_SOURCE_VALUE; SEMANTICS_UNRESOLVED
8,P2-EXTRAS2-009,NONZERO_EXTRAS2_WITH_RECORDED_OTHER,55078,1,20,124,71727,5,other,1,0,PRESERVE_SOURCE_VALUE; SEMANTICS_UNRESOLVED


New extras_2 exceptions: 9
Total Phase 2 exceptions: 10
PASS: Nine extras_2 exceptions documented; source values preserved.


#### Validate the Phase 2 exception register

In [43]:
# Standard schema for all Phase 2 exceptions.
exception_columns = [
    "exception_id",
    "exception_type",
    "dataset",
    "match_id",
    "affected_records",
    "field",
    "observation",
    "treatment",
    "status",
    "innings",
    "overid",
    "innings_ball_num",
    "bowler_id",
    "extras1_runs",
    "extras_type",
    "extras_2_runs",
    "runs"
]

# Normalize the original XI exception.
xi_exception = {
    "exception_id": "P2-XI-001",
    "exception_type": "MISSING_PLAYED_BATTING_ORDER",
    "dataset": "playing_eleven",
    "match_id": 48512,
    "affected_records": 11,
    "field": "batting_order",
    "observation": (
        "11 played registrations have missing batting_order values."
    ),
    "treatment": (
        "Preserve missing values. Do not infer or impute batting order."
    ),
    "status": "Documented - Unresolved"
}

# Normalize the nine extras exceptions.
extras_exceptions = df_extras2_exceptions.rename(columns={
    "exception_id": "exception_id",
    "exception_type": "exception_type",
    "match_id": "match_id",
    "innings": "innings",
    "overid": "overid",
    "innings_ball_num": "innings_ball_num",
    "bowler_id": "bowler_id",
    "extras1_runs": "extras1_runs",
    "extras_type": "extras_type",
    "extras_2_runs": "extras_2_runs",
    "runs": "runs",
    "treatment": "treatment"
}).copy()

extras_exceptions["dataset"] = "ball_by_ball"
extras_exceptions["affected_records"] = 1
extras_exceptions["field"] = "extras_2_runs"
extras_exceptions["observation"] = (
    "Nonzero extras_2_runs recorded with extras1_runs=5 "
    "and extras_type='other'; semantics unresolved."
)
extras_exceptions["status"] = "Documented - Unresolved"

# Combine into one consistent register.
phase2_exceptions = pd.concat(
    [
        pd.DataFrame([xi_exception]),
        extras_exceptions
    ],
    ignore_index=True
).reindex(columns=exception_columns)

# Validate.
print("REGISTER SHAPE:", phase2_exceptions.shape)

print("\nEXCEPTION COUNTS")
display(
    phase2_exceptions["exception_type"]
    .value_counts(dropna=False)
    .rename_axis("exception_type")
    .reset_index(name="count")
)

print("\nMISSING VALUES")
display(
    phase2_exceptions.isna()
    .sum()
    .rename_axis("column")
    .reset_index(name="missing_count")
    .query("missing_count > 0")
)

assert len(phase2_exceptions) == 10
assert phase2_exceptions["exception_id"].is_unique
assert phase2_exceptions["exception_type"].notna().all()
assert phase2_exceptions["dataset"].notna().all()
assert phase2_exceptions["treatment"].notna().all()

print("\nPASS: All 10 exceptions use a standardized schema.")

REGISTER SHAPE: (10, 17)

EXCEPTION COUNTS


,exception_type,count
0,NONZERO_EXTRAS2_WITH_RECORDED_OTHER,9
1,MISSING_PLAYED_BATTING_ORDER,1



MISSING VALUES


,column,missing_count
9,innings,1
10,overid,1
11,innings_ball_num,1
12,bowler_id,1
13,extras1_runs,1
14,extras_type,1
15,extras_2_runs,1
16,runs,1



PASS: All 10 exceptions use a standardized schema.


In [44]:
print("EXCEPTION REGISTER SHAPE")
print(phase2_exceptions.shape)

print("\nEXCEPTION COUNTS BY TYPE")
display(
    phase2_exceptions["exception_type"]
    .value_counts(dropna=False)
    .rename_axis("exception_type")
    .reset_index(name="exception_count")
)

print("\nMISSING VALUES BY COLUMN")
display(
    phase2_exceptions.isna()
    .sum()
    .rename_axis("column")
    .reset_index(name="missing_count")
    .query("missing_count > 0")
)

print("\nDUPLICATE EXCEPTION IDS")
print(
    "Duplicate IDs:",
    phase2_exceptions["exception_id"].duplicated().sum()
)

print("\nREGISTER")
display(phase2_exceptions)

assert phase2_exceptions["exception_id"].is_unique
assert len(phase2_exceptions) == 10
assert phase2_exceptions["exception_type"].notna().all()

print("\nPASS: Exception IDs are unique and all 10 exceptions are retained.")

EXCEPTION REGISTER SHAPE
(10, 17)

EXCEPTION COUNTS BY TYPE


,exception_type,exception_count
0,NONZERO_EXTRAS2_WITH_RECORDED_OTHER,9
1,MISSING_PLAYED_BATTING_ORDER,1



MISSING VALUES BY COLUMN


,column,missing_count
9,innings,1
10,overid,1
11,innings_ball_num,1
12,bowler_id,1
13,extras1_runs,1
14,extras_type,1
15,extras_2_runs,1
16,runs,1



DUPLICATE EXCEPTION IDS
Duplicate IDs: 0

REGISTER


,exception_id,exception_type,dataset,match_id,affected_records,field,observation,treatment,status,innings,overid,innings_ball_num,bowler_id,extras1_runs,extras_type,extras_2_runs,runs
0,P2-XI-001,MISSING_PLAYED_BATTING_ORDER,playing_eleven,48512,11,batting_order,11 played registrations have missing batting_o...,Preserve missing values. Do not infer or imput...,Documented - Unresolved,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,P2-EXTRAS2-001,NONZERO_EXTRAS2_WITH_RECORDED_OTHER,ball_by_ball,13614,1,extras_2_runs,Nonzero extras_2_runs recorded with extras1_ru...,PRESERVE_SOURCE_VALUE; SEMANTICS_UNRESOLVED,Documented - Unresolved,1.0,18.0,111.0,829.0,5.0,other,4.0,0.0
2,P2-EXTRAS2-002,NONZERO_EXTRAS2_WITH_RECORDED_OTHER,ball_by_ball,20234,1,extras_2_runs,Nonzero extras_2_runs recorded with extras1_ru...,PRESERVE_SOURCE_VALUE; SEMANTICS_UNRESOLVED,Documented - Unresolved,2.0,13.0,80.0,2261.0,5.0,other,4.0,0.0
3,P2-EXTRAS2-003,NONZERO_EXTRAS2_WITH_RECORDED_OTHER,ball_by_ball,20243,1,extras_2_runs,Nonzero extras_2_runs recorded with extras1_ru...,PRESERVE_SOURCE_VALUE; SEMANTICS_UNRESOLVED,Documented - Unresolved,2.0,19.0,121.0,5809.0,5.0,other,4.0,0.0
4,P2-EXTRAS2-004,NONZERO_EXTRAS2_WITH_RECORDED_OTHER,ball_by_ball,21583,1,extras_2_runs,Nonzero extras_2_runs recorded with extras1_ru...,PRESERVE_SOURCE_VALUE; SEMANTICS_UNRESOLVED,Documented - Unresolved,1.0,20.0,119.0,1566.0,5.0,other,1.0,0.0
5,P2-EXTRAS2-005,NONZERO_EXTRAS2_WITH_RECORDED_OTHER,ball_by_ball,37278,1,extras_2_runs,Nonzero extras_2_runs recorded with extras1_ru...,PRESERVE_SOURCE_VALUE; SEMANTICS_UNRESOLVED,Documented - Unresolved,1.0,12.0,69.0,2681.0,5.0,other,4.0,0.0
6,P2-EXTRAS2-006,NONZERO_EXTRAS2_WITH_RECORDED_OTHER,ball_by_ball,40038,1,extras_2_runs,Nonzero extras_2_runs recorded with extras1_ru...,PRESERVE_SOURCE_VALUE; SEMANTICS_UNRESOLVED,Documented - Unresolved,1.0,19.0,114.0,1566.0,5.0,other,1.0,0.0
7,P2-EXTRAS2-007,NONZERO_EXTRAS2_WITH_RECORDED_OTHER,ball_by_ball,40695,1,extras_2_runs,Nonzero extras_2_runs recorded with extras1_ru...,PRESERVE_SOURCE_VALUE; SEMANTICS_UNRESOLVED,Documented - Unresolved,1.0,18.0,115.0,3298.0,5.0,other,4.0,0.0
8,P2-EXTRAS2-008,NONZERO_EXTRAS2_WITH_RECORDED_OTHER,ball_by_ball,48467,1,extras_2_runs,Nonzero extras_2_runs recorded with extras1_ru...,PRESERVE_SOURCE_VALUE; SEMANTICS_UNRESOLVED,Documented - Unresolved,1.0,20.0,126.0,42548.0,5.0,other,1.0,0.0
9,P2-EXTRAS2-009,NONZERO_EXTRAS2_WITH_RECORDED_OTHER,ball_by_ball,55078,1,extras_2_runs,Nonzero extras_2_runs recorded with extras1_ru...,PRESERVE_SOURCE_VALUE; SEMANTICS_UNRESOLVED,Documented - Unresolved,1.0,20.0,124.0,71727.0,5.0,other,1.0,0.0



PASS: Exception IDs are unique and all 10 exceptions are retained.


### Save the exception register and classification outputs

In [45]:
# Save Phase 2 audit artifacts.
phase2_exceptions.to_csv(
    "/content/phase2_exception_register.csv",
    index=False
)

df_delivery_classified_v2.to_csv(
    "/content/phase2_eligible_regulation_deliveries_classified.csv",
    index=False
)

df_unexplained_over_full.to_csv(
    "/content/phase2_unexplained_extended_over_register.csv",
    index=False
)

print("Saved files:")
print("- phase2_exception_register.csv")
print("- phase2_eligible_regulation_deliveries_classified.csv")
print("- phase2_unexplained_extended_over_register.csv")

print("\nRow-count checks:")
print("Exceptions:", len(phase2_exceptions))
print("Classified regulation deliveries:", len(df_delivery_classified_v2))
print("Unexplained extended overs:", len(df_unexplained_over_full))

assert len(phase2_exceptions) == 10
assert len(df_delivery_classified_v2) == 192235
assert len(df_unexplained_over_full) == 147

print("\nPASS: Phase 2 audit artifacts saved and validated.")

Saved files:
- phase2_exception_register.csv
- phase2_eligible_regulation_deliveries_classified.csv
- phase2_unexplained_extended_over_register.csv

Row-count checks:
Exceptions: 10
Classified regulation deliveries: 192235
Unexplained extended overs: 147

PASS: Phase 2 audit artifacts saved and validated.


# Audit dismissal-field consistency

In [48]:
# DISMISSAL EXCEPTION INSPECTION — SELF-CONTAINED CELL

import pandas as pd

# 1. Verify required source columns.
required_columns = [
    "match_id",
    "innings",
    "overid",
    "innings_ball_num",
    "is_wicket",
    "dismissal_type",
    "dismissed_player_id"
]

missing_columns = [
    col for col in required_columns
    if col not in df_eligible_deliveries.columns
]

if missing_columns:
    raise KeyError(f"Missing required columns: {missing_columns}")

# 2. Create fresh audit DataFrames.
df_dismissal_audit = df_eligible_deliveries.copy()

# 3. Identify the two exception groups.
df_wicket_missing_dismissed_player = df_dismissal_audit[
    (df_dismissal_audit["is_wicket"] == True) &
    (df_dismissal_audit["dismissed_player_id"].isna())
].copy()

df_nonwicket_with_dismissed_player = df_dismissal_audit[
    (df_dismissal_audit["is_wicket"] == False) &
    (df_dismissal_audit["dismissed_player_id"].notna())
].copy()

# 4. Select only columns available in the dataset.
candidate_columns = [
    "match_id",
    "innings",
    "overid",
    "innings_ball_num",
    "over_ball_num",
    "is_wicket",
    "dismissal_type",
    "dismissed_player_id",
    "striker_id",
    "non_striker_id",
    "bowler_id"
]

available_columns = [
    col for col in candidate_columns
    if col in df_dismissal_audit.columns
]

# 5. Display exception records.
print("WICKET RECORD MISSING DISMISSED PLAYER ID")
display(
    df_wicket_missing_dismissed_player[available_columns]
)

print("NON-WICKET RECORDS WITH DISMISSED PLAYER ID")
display(
    df_nonwicket_with_dismissed_player[available_columns]
)

# 6. Validate counts.
print("EXCEPTION COUNTS")
print("Wicket without dismissed player ID:",
      len(df_wicket_missing_dismissed_player))
print("Non-wicket with dismissed player ID:",
      len(df_nonwicket_with_dismissed_player))

assert len(df_wicket_missing_dismissed_player) == 1
assert len(df_nonwicket_with_dismissed_player) == 11

print("PASS: Dismissal exception inspection completed.")

WICKET RECORD MISSING DISMISSED PLAYER ID


,match_id,innings,overid,innings_ball_num,over_ball_num,is_wicket,dismissal_type,dismissed_player_id,striker_id,non_striker_id,bowler_id
127899,40709,2,19,117,5,True,Retire Out,NaN,3330,7897,2847


NON-WICKET RECORDS WITH DISMISSED PLAYER ID


,match_id,innings,overid,innings_ball_num,over_ball_num,is_wicket,dismissal_type,dismissed_player_id,striker_id,non_striker_id,bowler_id
8245,14114,2,14,80,1,False,NaN,1440.0,1440,1568,3926
23631,15610,1,6,38,6,False,NaN,6433.0,6433,2533,22134
25855,16064,2,2,10,4,False,NaN,126.0,126,1699,3275
47750,20188,1,11,64,1,False,NaN,2003.0,142,2003,305
61016,20243,1,16,99,6,False,NaN,4219.0,2639,4219,5939
63576,20880,2,11,65,3,False,NaN,2533.0,2533,3544,27516
77112,28964,1,10,65,6,False,NaN,1568.0,1568,4847,19145
79418,29926,2,9,54,4,False,NaN,6931.0,6931,4900,7967
98952,31709,1,13,80,3,False,NaN,3024.0,20327,3024,2526
131669,40725,2,6,36,4,False,NaN,2258.0,2258,6819,46066


EXCEPTION COUNTS
Wicket without dismissed player ID: 1
Non-wicket with dismissed player ID: 11
PASS: Dismissal exception inspection completed.


### Validate dismissed-player ID relationships

In [49]:
# Create a separate inspection copy of the 11 non-wicket exceptions.
df_nonwicket_id_analysis = df_nonwicket_with_dismissed_player.copy()

# Compare dismissed_player_id with both batter IDs.
df_nonwicket_id_analysis["matches_striker"] = (
    df_nonwicket_id_analysis["dismissed_player_id"]
    == df_nonwicket_id_analysis["striker_id"]
)

df_nonwicket_id_analysis["matches_non_striker"] = (
    df_nonwicket_id_analysis["dismissed_player_id"]
    == df_nonwicket_id_analysis["non_striker_id"]
)

# Classify the observed relationship without assigning dismissal meaning.
conditions = [
    df_nonwicket_id_analysis["matches_striker"]
    & df_nonwicket_id_analysis["matches_non_striker"],

    df_nonwicket_id_analysis["matches_striker"],

    df_nonwicket_id_analysis["matches_non_striker"]
]

choices = [
    "MATCHES_BOTH",
    "MATCHES_STRIKER",
    "MATCHES_NON_STRIKER"
]

df_nonwicket_id_analysis["player_id_relationship"] = (
    np.select(
        conditions,
        choices,
        default="MATCHES_NEITHER"
    )
)

print("NON-WICKET DISMISSED PLAYER ID RELATIONSHIPS")
display(
    df_nonwicket_id_analysis[
        [
            "match_id",
            "innings",
            "overid",
            "innings_ball_num",
            "dismissed_player_id",
            "striker_id",
            "non_striker_id",
            "player_id_relationship"
        ]
    ]
)

print("\nRELATIONSHIP COUNTS")
display(
    df_nonwicket_id_analysis["player_id_relationship"]
    .value_counts()
    .rename_axis("relationship")
    .reset_index(name="record_count")
)

# Inspect the Retire Out exception separately.
print("\nRETIRE OUT RECORD WITH MISSING DISMISSED PLAYER ID")
display(
    df_wicket_missing_dismissed_player[
        [
            "match_id",
            "innings",
            "overid",
            "innings_ball_num",
            "dismissal_type",
            "dismissed_player_id",
            "striker_id",
            "non_striker_id",
            "bowler_id"
        ]
    ]
)

assert len(df_nonwicket_id_analysis) == 11
assert len(df_wicket_missing_dismissed_player) == 1

print("\nPASS: Player-ID relationships inspected; source values unchanged.")

NON-WICKET DISMISSED PLAYER ID RELATIONSHIPS


,match_id,innings,overid,innings_ball_num,dismissed_player_id,striker_id,non_striker_id,player_id_relationship
8245,14114,2,14,80,1440.0,1440,1568,MATCHES_STRIKER
23631,15610,1,6,38,6433.0,6433,2533,MATCHES_STRIKER
25855,16064,2,2,10,126.0,126,1699,MATCHES_STRIKER
47750,20188,1,11,64,2003.0,142,2003,MATCHES_NON_STRIKER
61016,20243,1,16,99,4219.0,2639,4219,MATCHES_NON_STRIKER
63576,20880,2,11,65,2533.0,2533,3544,MATCHES_STRIKER
77112,28964,1,10,65,1568.0,1568,4847,MATCHES_STRIKER
79418,29926,2,9,54,6931.0,6931,4900,MATCHES_STRIKER
98952,31709,1,13,80,3024.0,20327,3024,MATCHES_NON_STRIKER
131669,40725,2,6,36,2258.0,2258,6819,MATCHES_STRIKER



RELATIONSHIP COUNTS


,relationship,record_count
0,MATCHES_STRIKER,8
1,MATCHES_NON_STRIKER,3



RETIRE OUT RECORD WITH MISSING DISMISSED PLAYER ID


,match_id,innings,overid,innings_ball_num,dismissal_type,dismissed_player_id,striker_id,non_striker_id,bowler_id
127899,40709,2,19,117,Retire Out,NaN,3330,7897,2847



PASS: Player-ID relationships inspected; source values unchanged.


### Add the 12 dismissal exceptions to the register

In [50]:
# Build standardized dismissal exception records.

dismissal_exceptions = []

# Exception 1: Retire Out with missing dismissed_player_id.
retire_row = df_wicket_missing_dismissed_player.iloc[0]

dismissal_exceptions.append({
    "exception_id": "P2-DISM-001",
    "exception_type": "WICKET_MISSING_DISMISSED_PLAYER_ID",
    "dataset": "ball_by_ball",
    "match_id": int(retire_row["match_id"]),
    "affected_records": 1,
    "field": "dismissed_player_id",
    "observation": (
        "Retire Out recorded with missing dismissed_player_id; "
        f"striker_id={int(retire_row['striker_id'])}."
    ),
    "treatment": "Preserve missing value; do not infer dismissed_player_id.",
    "status": "Documented - Unresolved"
})

# Exceptions 2–12: Non-wicket records with dismissed_player_id.
for i, (_, row) in enumerate(
    df_nonwicket_id_analysis.iterrows(),
    start=2
):
    dismissal_exceptions.append({
        "exception_id": f"P2-DISM-{i:03d}",
        "exception_type": "NON_WICKET_WITH_DISMISSED_PLAYER_ID",
        "dataset": "ball_by_ball",
        "match_id": int(row["match_id"]),
        "affected_records": 1,
        "field": "dismissed_player_id",
        "observation": (
            f"is_wicket=False; dismissed_player_id "
            f"{row['player_id_relationship'].lower()}."
        ),
        "treatment": (
            "Preserve source values; do not reinterpret wicket status "
            "or dismissed player."
        ),
        "status": "Documented - Unresolved"
    })

# Convert to the existing standardized schema.
df_dismissal_exceptions = pd.DataFrame(
    dismissal_exceptions
).reindex(columns=phase2_exceptions.columns)

# Add the records to the register.
phase2_exceptions = pd.concat(
    [phase2_exceptions, df_dismissal_exceptions],
    ignore_index=True
)

# Validate the updated register.
print("DISMISSAL EXCEPTIONS ADDED:", len(df_dismissal_exceptions))
print("TOTAL EXCEPTIONS:", len(phase2_exceptions))

display(
    phase2_exceptions[
        ["exception_id", "exception_type", "match_id",
         "field", "observation", "treatment", "status"]
    ]
)

assert len(df_dismissal_exceptions) == 12
assert len(phase2_exceptions) == 22
assert phase2_exceptions["exception_id"].is_unique

print("PASS: 12 dismissal exceptions documented; source data unchanged.")

DISMISSAL EXCEPTIONS ADDED: 12
TOTAL EXCEPTIONS: 22


,exception_id,exception_type,match_id,field,observation,treatment,status
0,P2-XI-001,MISSING_PLAYED_BATTING_ORDER,48512,batting_order,11 played registrations have missing batting_o...,Preserve missing values. Do not infer or imput...,Documented - Unresolved
1,P2-EXTRAS2-001,NONZERO_EXTRAS2_WITH_RECORDED_OTHER,13614,extras_2_runs,Nonzero extras_2_runs recorded with extras1_ru...,PRESERVE_SOURCE_VALUE; SEMANTICS_UNRESOLVED,Documented - Unresolved
2,P2-EXTRAS2-002,NONZERO_EXTRAS2_WITH_RECORDED_OTHER,20234,extras_2_runs,Nonzero extras_2_runs recorded with extras1_ru...,PRESERVE_SOURCE_VALUE; SEMANTICS_UNRESOLVED,Documented - Unresolved
3,P2-EXTRAS2-003,NONZERO_EXTRAS2_WITH_RECORDED_OTHER,20243,extras_2_runs,Nonzero extras_2_runs recorded with extras1_ru...,PRESERVE_SOURCE_VALUE; SEMANTICS_UNRESOLVED,Documented - Unresolved
4,P2-EXTRAS2-004,NONZERO_EXTRAS2_WITH_RECORDED_OTHER,21583,extras_2_runs,Nonzero extras_2_runs recorded with extras1_ru...,PRESERVE_SOURCE_VALUE; SEMANTICS_UNRESOLVED,Documented - Unresolved
5,P2-EXTRAS2-005,NONZERO_EXTRAS2_WITH_RECORDED_OTHER,37278,extras_2_runs,Nonzero extras_2_runs recorded with extras1_ru...,PRESERVE_SOURCE_VALUE; SEMANTICS_UNRESOLVED,Documented - Unresolved
6,P2-EXTRAS2-006,NONZERO_EXTRAS2_WITH_RECORDED_OTHER,40038,extras_2_runs,Nonzero extras_2_runs recorded with extras1_ru...,PRESERVE_SOURCE_VALUE; SEMANTICS_UNRESOLVED,Documented - Unresolved
7,P2-EXTRAS2-007,NONZERO_EXTRAS2_WITH_RECORDED_OTHER,40695,extras_2_runs,Nonzero extras_2_runs recorded with extras1_ru...,PRESERVE_SOURCE_VALUE; SEMANTICS_UNRESOLVED,Documented - Unresolved
8,P2-EXTRAS2-008,NONZERO_EXTRAS2_WITH_RECORDED_OTHER,48467,extras_2_runs,Nonzero extras_2_runs recorded with extras1_ru...,PRESERVE_SOURCE_VALUE; SEMANTICS_UNRESOLVED,Documented - Unresolved
9,P2-EXTRAS2-009,NONZERO_EXTRAS2_WITH_RECORDED_OTHER,55078,extras_2_runs,Nonzero extras_2_runs recorded with extras1_ru...,PRESERVE_SOURCE_VALUE; SEMANTICS_UNRESOLVED,Documented - Unresolved


PASS: 12 dismissal exceptions documented; source data unchanged.


In [51]:
# Dismissal type and ID consistency audit.
df_dismissal_id_audit = df_eligible_deliveries[
    df_eligible_deliveries["is_wicket"] == True
].copy()

print("DISMISSAL TYPE × DISMISSAL TYPE ID")
display(
    df_dismissal_id_audit.groupby(
        ["dismissal_type", "dismissal_type_id"],
        dropna=False
    )
    .size()
    .reset_index(name="record_count")
    .sort_values(
        ["dismissal_type", "record_count"],
        ascending=[True, False]
    )
)

print("\nSPELLING VARIANT RECORDS")
spelling_variants = [
    "Obstracting the field",
    "Obstructing the field"
]

df_spelling_variants = df_dismissal_id_audit[
    df_dismissal_id_audit["dismissal_type"].isin(spelling_variants)
].copy()

display(
    df_spelling_variants[
        [
            "match_id",
            "innings",
            "overid",
            "innings_ball_num",
            "dismissal_type",
            "dismissal_type_id",
            "dismissed_player_id",
            "striker_id",
            "non_striker_id",
            "bowler_id"
        ]
    ]
)

print("\nVARIANT SUMMARY")
display(
    df_spelling_variants.groupby(
        ["dismissal_type", "dismissal_type_id"],
        dropna=False
    )
    .size()
    .reset_index(name="record_count")
)

assert len(df_spelling_variants) == 2

print("\nPASS: Dismissal spelling variants extracted.")

DISMISSAL TYPE × DISMISSAL TYPE ID


,dismissal_type,dismissal_type_id,record_count
0,Bowled,1.0,1676
1,Caught,2.0,7269
2,Hit Wicket,6.0,17
3,LBW,3.0,663
4,Obstracting the field,10.0,1
5,Obstructing the field,10.0,1
6,Retire Out,9.0,6
7,RunOut,4.0,569
8,Stump,5.0,198



SPELLING VARIANT RECORDS


,match_id,innings,overid,innings_ball_num,dismissal_type,dismissal_type_id,dismissed_player_id,striker_id,non_striker_id,bowler_id
96293,31279,2,16,100,Obstracting the field,10.0,152.0,152,6059,3653
185026,50359,1,5,32,Obstructing the field,10.0,26906.0,26906,5939,58246



VARIANT SUMMARY


,dismissal_type,dismissal_type_id,record_count
0,Obstracting the field,10.0,1
1,Obstructing the field,10.0,1



PASS: Dismissal spelling variants extracted.


### Create normalized dismissal labels

In [52]:
# Create a separate classified copy of eligible deliveries.
df_dismissal_classified = df_eligible_deliveries.copy()

# Preserve original dismissal_type.
df_dismissal_classified["dismissal_type_original"] = (
    df_dismissal_classified["dismissal_type"]
)

# Create normalized analytical label.
dismissal_label_map = {
    "Obstracting the field": "Obstructing the field"
}

df_dismissal_classified["dismissal_type_normalized"] = (
    df_dismissal_classified["dismissal_type"]
    .replace(dismissal_label_map)
)

# Validate original and normalized values.
print("ORIGINAL VS NORMALIZED DISMISSAL LABELS")
display(
    df_dismissal_classified[
        df_dismissal_classified["dismissal_type"].isin(
            ["Obstracting the field", "Obstructing the field"]
        )
    ][
        [
            "match_id",
            "dismissal_type_id",
            "dismissal_type_original",
            "dismissal_type_normalized"
        ]
    ]
)

print("\nNORMALIZED DISMISSAL COUNTS")
display(
    df_dismissal_classified[
        df_dismissal_classified["is_wicket"] == True
    ]["dismissal_type_normalized"]
    .value_counts(dropna=False)
    .rename_axis("dismissal_type_normalized")
    .reset_index(name="record_count")
)

# Assertions.
assert (
    df_dismissal_classified["dismissal_type_original"]
    .equals(df_dismissal_classified["dismissal_type"])
)

assert (
    df_dismissal_classified["dismissal_type_normalized"]
    .eq("Obstructing the field")
    .sum() == 2
)

print("\nPASS: Two spelling variants normalized; original labels preserved.")

ORIGINAL VS NORMALIZED DISMISSAL LABELS


,match_id,dismissal_type_id,dismissal_type_original,dismissal_type_normalized
96293,31279,10.0,Obstracting the field,Obstructing the field
185026,50359,10.0,Obstructing the field,Obstructing the field



NORMALIZED DISMISSAL COUNTS


,dismissal_type_normalized,record_count
0,Caught,7269
1,Bowled,1676
2,LBW,663
3,RunOut,569
4,Stump,198
5,Hit Wicket,17
6,Retire Out,6
7,Obstructing the field,2



PASS: Two spelling variants normalized; original labels preserved.


### Document the spelling normalization

In [53]:
# Document dismissal spelling variation.

spelling_exception = pd.DataFrame([{
    "exception_id": "P2-DISM-013",
    "exception_type": "DISMISSAL_LABEL_SPELLING_VARIANT",
    "dataset": "ball_by_ball",
    "match_id": 31279,
    "affected_records": 2,
    "field": "dismissal_type",
    "observation": (
        "Two source labels, 'Obstracting the field' and "
        "'Obstructing the field', share dismissal_type_id=10."
    ),
    "treatment": (
        "Normalize both to 'Obstructing the field' in the "
        "analytical copy; preserve original source labels."
    ),
    "status": "Documented - Normalized"
}]).reindex(columns=phase2_exceptions.columns)

phase2_exceptions = pd.concat(
    [phase2_exceptions, spelling_exception],
    ignore_index=True
)

assert len(phase2_exceptions) == 23
assert phase2_exceptions["exception_id"].is_unique

print("New spelling exception documented:", len(spelling_exception))
print("Total Phase 2 exceptions:", len(phase2_exceptions))

display(
    phase2_exceptions[
        phase2_exceptions["exception_id"] == "P2-DISM-013"
    ]
)

print("PASS: Dismissal spelling normalization documented.")

New spelling exception documented: 1
Total Phase 2 exceptions: 23


,exception_id,exception_type,dataset,match_id,affected_records,field,observation,treatment,status,innings,overid,innings_ball_num,bowler_id,extras1_runs,extras_type,extras_2_runs,runs
22,P2-DISM-013,DISMISSAL_LABEL_SPELLING_VARIANT,ball_by_ball,31279,2,dismissal_type,"Two source labels, 'Obstracting the field' and...",Normalize both to 'Obstructing the field' in t...,Documented - Normalized,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


PASS: Dismissal spelling normalization documented.


### Audit player identity consistency

In [55]:
# PLAYER IDENTITY CONSISTENCY AUDIT

# 1. Identify player-related columns.
player_id_columns = [
    col for col in [
        "striker_id",
        "non_striker_id",
        "bowler_id",
        "dismissed_player_id"
    ]
    if col in df_eligible_deliveries.columns
]

print("PLAYER ID COLUMNS:", player_id_columns)

# 2. Use the existing raw mapping DataFrame.
print("MAPPING TABLE COLUMNS:")
print(df_mapping_raw.columns.tolist())

# 3. Verify player_id exists.
assert "player_id" in df_mapping_raw.columns, (
    "player_id column not found in mapping table."
)

known_player_ids = set(
    df_mapping_raw["player_id"].dropna().unique()
)

# 4. Compare each delivery role against mapping IDs.
identity_results = []

for col in player_id_columns:
    observed_ids = set(
        df_eligible_deliveries[col].dropna().unique()
    )

    unmatched_ids = observed_ids - known_player_ids

    identity_results.append({
        "delivery_role": col,
        "distinct_observed_ids": len(observed_ids),
        "distinct_matched_ids": len(observed_ids & known_player_ids),
        "distinct_unmatched_ids": len(unmatched_ids),
        "unmatched_id_sample": sorted(unmatched_ids)[:20]
    })

df_identity_audit = pd.DataFrame(identity_results)

display(df_identity_audit)

print("\nPASS: Player identity coverage compared against mapping IDs.")

PLAYER ID COLUMNS: ['striker_id', 'non_striker_id', 'bowler_id', 'dismissed_player_id']
MAPPING TABLE COLUMNS:
['comp_id', 'comp_name', 'team_id', 'team_name', 'player_id', 'player_name']


,delivery_role,distinct_observed_ids,distinct_matched_ids,distinct_unmatched_ids,unmatched_id_sample
0,striker_id,824,824,0,[]
1,non_striker_id,806,806,0,[]
2,bowler_id,596,596,0,[]
3,dismissed_player_id,758,758,0,[]



PASS: Player identity coverage compared against mapping IDs.


### Check mapping consistency

In [56]:
# PLAYER MAPPING CONSISTENCY AUDIT

# 1. Count distinct competitions and teams per player ID.
player_mapping_profile = (
    df_mapping_raw.groupby("player_id")
    .agg(
        player_name_variants=("player_name", "nunique"),
        team_count=("team_id", "nunique"),
        competition_count=("comp_id", "nunique"),
        mapping_rows=("player_id", "size")
    )
    .reset_index()
)

# 2. Summarize mapping patterns.
print("PLAYER MAPPING PROFILE")
display(
    player_mapping_profile[
        ["player_name_variants", "team_count",
         "competition_count", "mapping_rows"]
    ].describe().round(2)
)

# 3. Identify players associated with multiple names.
print("PLAYER IDs WITH MULTIPLE NAMES")
display(
    player_mapping_profile[
        player_mapping_profile["player_name_variants"] > 1
    ].sort_values("player_name_variants", ascending=False)
)

# 4. Identify players mapped to multiple teams.
print("PLAYER IDs WITH MULTIPLE TEAMS")
display(
    player_mapping_profile[
        player_mapping_profile["team_count"] > 1
    ].sort_values("team_count", ascending=False)
)

# 5. Identify duplicate player-team-competition records.
duplicate_mapping_rows = df_mapping_raw[
    df_mapping_raw.duplicated(
        subset=["comp_id", "team_id", "player_id"],
        keep=False
    )
].sort_values(["player_id", "comp_id", "team_id"])

print("DUPLICATE PLAYER-COMPETITION-TEAM MAPPING ROWS")
display(duplicate_mapping_rows)

print("Duplicate mapping rows:", len(duplicate_mapping_rows))

PLAYER MAPPING PROFILE


,player_name_variants,team_count,competition_count,mapping_rows
count,1217.0,1217.00,1217.00,1217.00
mean,1.0,1.47,5.27,5.27
std,0.0,0.81,5.83,5.83
min,1.0,1.00,1.00,1.00
25%,1.0,1.00,1.00,1.00
50%,1.0,1.00,3.00,3.00
75%,1.0,2.00,7.00,7.00
max,1.0,6.00,33.00,33.00


PLAYER IDs WITH MULTIPLE NAMES


,player_id,player_name_variants,team_count,competition_count,mapping_rows


PLAYER IDs WITH MULTIPLE TEAMS


,player_id,player_name_variants,team_count,competition_count,mapping_rows
182,2847,1,6,6,6
573,6726,1,5,28,28
412,4900,1,5,24,24
572,6725,1,5,18,18
365,4692,1,5,19,19
...,...,...,...,...,...
1059,34485,1,2,12,12
1078,35318,1,2,2,2
1182,56685,1,2,9,9
1189,58246,1,2,7,7


DUPLICATE PLAYER-COMPETITION-TEAM MAPPING ROWS


,comp_id,comp_name,team_id,team_name,player_id,player_name


Duplicate mapping rows: 0


### Validate player-team mapping against delivery participation

In [57]:
# INSPECT DELIVERY AND MAPPING TEAM FIELDS

print("BALL-BY-BALL TEAM AND PLAYER COLUMNS")

team_player_columns = [
    col for col in df_eligible_deliveries.columns
    if any(term in col.lower() for term in ["team", "player", "striker", "bowler"])
]

print(team_player_columns)

print("\nMAPPING TABLE SAMPLE")
display(df_mapping_raw.head(10))

print("\nMATCH TABLE TEAM COLUMNS")
print([
    col for col in df_matches_raw.columns
    if "team" in col.lower() or "match_id" in col.lower()
])

print("\nPLAYING XI COLUMNS")
print(df_playing_eleven_raw.columns.tolist())

BALL-BY-BALL TEAM AND PLAYER COLUMNS
['batting_team_id', 'bowling_team_id', 'striker_id', 'striker_name', 'non_striker_id', 'non_striker_name', 'bowler_id', 'bowler_name', 'dismissed_player_id']

MAPPING TABLE SAMPLE


,comp_id,comp_name,team_id,team_name,player_id,player_name
0,1331,England in West Indies T20I Series 2022,22,England,478,Eoin Morgan
1,1331,England in West Indies T20I Series 2022,22,England,479,Adil Rashid
2,1331,England in West Indies T20I Series 2022,22,England,722,Liam Dawson
3,1331,England in West Indies T20I Series 2022,22,England,786,Chris Jordan
4,1331,England in West Indies T20I Series 2022,22,England,1187,Jason Roy
5,1331,England in West Indies T20I Series 2022,22,England,1210,James Vince
6,1331,England in West Indies T20I Series 2022,22,England,1556,David Payne
7,1331,England in West Indies T20I Series 2022,22,England,1596,Moeen Ali
8,1331,England in West Indies T20I Series 2022,22,England,1924,Sam Billings
9,1331,England in West Indies T20I Series 2022,22,England,2304,Tymal Mills



MATCH TABLE TEAM COLUMNS
['match_id', 'home_team_id', 'home_team', 'away_team_id', 'away_team', 'home_team_captain_id', 'away_team_captain_id', 'toss_win_team_id', 'win_team_id']

PLAYING XI COLUMNS
['match_id', 'team_id', 'player_id', 'batting_order', 'is_played', 'is_captain', 'is_wk', 'is_impact', 'is_impact_off']


### Player-to-team consistency audit

In [58]:
# PLAYER-TEAM-COMPETITION CONSISTENCY AUDIT

# 1. Attach competition ID to eligible deliveries.
df_team_identity = df_eligible_deliveries.merge(
    df_eligible_matches[["match_id", "comp_id"]],
    on="match_id",
    how="left",
    validate="many_to_one"
)

assert df_team_identity["comp_id"].notna().all()

# 2. Build the approved mapping key set.
mapping_keys = set(
    zip(
        df_mapping_raw["comp_id"],
        df_mapping_raw["team_id"],
        df_mapping_raw["player_id"]
    )
)

# 3. Define expected player-team relationships.
role_config = {
    "striker_id": "batting_team_id",
    "non_striker_id": "batting_team_id",
    "bowler_id": "bowling_team_id",
    "dismissed_player_id": "batting_team_id"
}

identity_results = []
identity_exceptions = []

# 4. Test each role.
for player_col, team_col in role_config.items():

    role_data = df_team_identity[
        ["match_id", "comp_id", team_col, player_col]
    ].dropna().copy()

    role_data["mapping_exists"] = [
        (comp, team, player) in mapping_keys
        for comp, team, player in zip(
            role_data["comp_id"],
            role_data[team_col],
            role_data[player_col]
        )
    ]

    unmatched = role_data[
        ~role_data["mapping_exists"]
    ].copy()

    identity_results.append({
        "delivery_role": player_col,
        "expected_team": team_col,
        "records_checked": len(role_data),
        "matched_records": int(role_data["mapping_exists"].sum()),
        "unmatched_records": len(unmatched),
        "distinct_unmatched_players": unmatched[player_col].nunique()
    })

    if len(unmatched) > 0:
        unmatched["delivery_role"] = player_col
        unmatched["expected_team"] = team_col
        unmatched = unmatched.rename(columns={
            player_col: "player_id",
            team_col: "expected_team_id"
        })

        identity_exceptions.append(unmatched)

# 5. Display summary.
df_team_identity_audit = pd.DataFrame(identity_results)

print("PLAYER-TEAM-COMPETITION MATCHING SUMMARY")
display(df_team_identity_audit)

# 6. Display unmatched records, if any.
if identity_exceptions:
    df_team_identity_exceptions = pd.concat(
        identity_exceptions,
        ignore_index=True
    )

    print("UNMATCHED PLAYER-TEAM RECORDS")
    display(df_team_identity_exceptions.head(50))
else:
    df_team_identity_exceptions = pd.DataFrame()
    print("No unmatched player-team-competition records.")

print("\nTOTAL UNMATCHED RECORDS:",
      len(df_team_identity_exceptions))

print("\nAUDIT COMPLETED")

PLAYER-TEAM-COMPETITION MATCHING SUMMARY


,delivery_role,expected_team,records_checked,matched_records,unmatched_records,distinct_unmatched_players
0,striker_id,batting_team_id,192261,192261,0,0
1,non_striker_id,batting_team_id,192261,192261,0,0
2,bowler_id,bowling_team_id,192261,192261,0,0
3,dismissed_player_id,batting_team_id,10410,10410,0,0


No unmatched player-team-competition records.

TOTAL UNMATCHED RECORDS: 0

AUDIT COMPLETED


### Delivery sequence integrity

In [59]:
# DELIVERY SEQUENCE INTEGRITY AUDIT

sequence_audit = (
    df_eligible_regulation_deliveries
    .groupby(["match_id", "innings"])
    .agg(
        delivery_records=("innings_ball_num", "size"),
        unique_innings_ball_num=("innings_ball_num", "nunique"),
        min_innings_ball_num=("innings_ball_num", "min"),
        max_innings_ball_num=("innings_ball_num", "max"),
        unique_over_ids=("overid", "nunique"),
        min_over_id=("overid", "min"),
        max_over_id=("overid", "max")
    )
    .reset_index()
)

sequence_audit["innings_ball_num_contiguous"] = (
    sequence_audit["delivery_records"]
    == sequence_audit["unique_innings_ball_num"]
) & (
    sequence_audit["max_innings_ball_num"]
    - sequence_audit["min_innings_ball_num"] + 1
    == sequence_audit["delivery_records"]
)

print("INNINGS SEQUENCE SUMMARY")
display(
    sequence_audit[
        [
            "delivery_records",
            "unique_innings_ball_num",
            "innings_ball_num_contiguous"
        ]
    ].describe(include="all")
)

print("\nNON-CONTIGUOUS OR DUPLICATED INNINGS SEQUENCES")
df_sequence_exceptions = sequence_audit[
    ~sequence_audit["innings_ball_num_contiguous"]
]

display(df_sequence_exceptions)

print("\nTOTAL INNINGS:", len(sequence_audit))
print("SEQUENCE EXCEPTIONS:", len(df_sequence_exceptions))

assert len(sequence_audit) > 0

print("\nPASS: Innings sequence integrity audit completed.")

INNINGS SEQUENCE SUMMARY


,delivery_records,unique_innings_ball_num,innings_ball_num_contiguous
count,1626.000000,1626.000000,1626
unique,NaN,NaN,1
top,NaN,NaN,True
freq,NaN,NaN,1626
mean,118.225707,118.225707,NaN
std,15.308121,15.308121,NaN
min,30.000000,30.000000,NaN
25%,118.000000,118.000000,NaN
50%,123.000000,123.000000,NaN
75%,126.000000,126.000000,NaN



NON-CONTIGUOUS OR DUPLICATED INNINGS SEQUENCES


,match_id,innings,delivery_records,unique_innings_ball_num,min_innings_ball_num,max_innings_ball_num,unique_over_ids,min_over_id,max_over_id,innings_ball_num_contiguous



TOTAL INNINGS: 1626
SEQUENCE EXCEPTIONS: 0

PASS: Innings sequence integrity audit completed.


### Validate over sequence structure

In [60]:
# OVER SEQUENCE STRUCTURE AUDIT

df_over_sequence = (
    df_eligible_regulation_deliveries
    .groupby(["match_id", "innings", "overid"], as_index=False)
    .agg(
        delivery_count=("innings_ball_num", "size"),
        unique_over_ball_num=("over_ball_num", "nunique"),
        min_over_ball_num=("over_ball_num", "min"),
        max_over_ball_num=("over_ball_num", "max"),
        min_innings_ball_num=("innings_ball_num", "min"),
        max_innings_ball_num=("innings_ball_num", "max")
    )
)

df_over_sequence["over_ball_num_contiguous"] = (
    df_over_sequence["delivery_count"]
    == df_over_sequence["unique_over_ball_num"]
) & (
    df_over_sequence["max_over_ball_num"]
    - df_over_sequence["min_over_ball_num"] + 1
    == df_over_sequence["delivery_count"]
)

print("OVER RECORD COUNT DISTRIBUTION")
display(
    df_over_sequence["delivery_count"]
    .value_counts()
    .sort_index()
    .rename_axis("delivery_count")
    .reset_index(name="over_count")
)

print("\nOVER BALL NUMBER CONTINUITY")
display(
    df_over_sequence["over_ball_num_contiguous"]
    .value_counts()
    .rename_axis("is_contiguous")
    .reset_index(name="over_count")
)

print("\nOVERS WITH NON-CONTIGUOUS OVER BALL NUMBERS")
df_over_sequence_exceptions = df_over_sequence[
    ~df_over_sequence["over_ball_num_contiguous"]
]

display(df_over_sequence_exceptions.head(30))

print("\nOVERS WITH MORE THAN SIX DELIVERY RECORDS")
display(
    df_over_sequence[
        df_over_sequence["delivery_count"] > 6
    ].head(20)
)

print("\nTOTAL OVERS:", len(df_over_sequence))
print("NON-CONTIGUOUS OVERS:", len(df_over_sequence_exceptions))

OVER RECORD COUNT DISTRIBUTION


,delivery_count,over_count
0,1,86
1,2,79
2,3,97
3,4,120
4,5,135
5,6,23954
6,7,5128
7,8,1048
8,9,218
9,10,41



OVER BALL NUMBER CONTINUITY


,is_contiguous,over_count
0,True,30921



OVERS WITH NON-CONTIGUOUS OVER BALL NUMBERS


,match_id,innings,overid,delivery_count,unique_over_ball_num,min_over_ball_num,max_over_ball_num,min_innings_ball_num,max_innings_ball_num,over_ball_num_contiguous



OVERS WITH MORE THAN SIX DELIVERY RECORDS


,match_id,innings,overid,delivery_count,unique_over_ball_num,min_over_ball_num,max_over_ball_num,min_innings_ball_num,max_innings_ball_num,over_ball_num_contiguous
6,13511,1,7,7,7,1,7,37,43,True
7,13511,1,8,8,8,1,8,44,51,True
47,13512,1,10,8,8,1,8,55,62,True
49,13512,1,12,7,7,1,7,69,75,True
51,13512,1,14,7,7,1,7,82,88,True
53,13512,1,16,7,7,1,7,95,101,True
54,13512,1,17,7,7,1,7,102,108,True
55,13512,1,18,7,7,1,7,109,115,True
56,13512,1,19,7,7,1,7,116,122,True
58,13512,2,1,7,7,1,7,1,7,True



TOTAL OVERS: 30921
NON-CONTIGUOUS OVERS: 0


### Inspect recorded delivery events in extended overs

In [61]:
# EXTENDED OVER EVENT COMPOSITION AUDIT

df_extended_over_events = df_eligible_regulation_deliveries.merge(
    df_over_sequence[
        ["match_id", "innings", "overid", "delivery_count"]
    ],
    on=["match_id", "innings", "overid"],
    how="left",
    validate="many_to_one"
)

df_extended_over_events = df_extended_over_events[
    df_extended_over_events["delivery_count"] > 6
].copy()

# Aggregate source event codes within each extended over.
df_extended_over_profile = (
    df_extended_over_events
    .groupby(["match_id", "innings", "overid"], as_index=False)
    .agg(
        delivery_count=("match_id", "size"),
        wide_records=("extras1_runs", lambda x: (x == 2).sum()),
        overstep_records=("extras1_runs", lambda x: (x == 1).sum()),
        other_records=("extras1_runs", lambda x: (x == 5).sum()),
        legbye_records=("extras1_runs", lambda x: (x == 3).sum()),
        bye_records=("extras1_runs", lambda x: (x == 4).sum()),
        no_extras_records=("extras1_runs", lambda x: (x == 6).sum()),
        extras2_nonzero=("extras_2_runs", lambda x: (x != 0).sum())
    )
)

# Classify only the observable evidence pattern.
df_extended_over_profile["recorded_wide_or_overstep"] = (
    (df_extended_over_profile["wide_records"] > 0) |
    (df_extended_over_profile["overstep_records"] > 0)
)

print("EXTENDED OVER COMPOSITION SUMMARY")
display(
    df_extended_over_profile.groupby(
        "recorded_wide_or_overstep"
    ).agg(
        over_count=("match_id", "size"),
        delivery_records=("delivery_count", "sum"),
        total_wides=("wide_records", "sum"),
        total_oversteps=("overstep_records", "sum"),
        total_other=("other_records", "sum"),
        total_legbyes=("legbye_records", "sum"),
        total_byes=("bye_records", "sum"),
        total_no_extras=("no_extras_records", "sum"),
        total_extras2_nonzero=("extras2_nonzero", "sum")
    ).reset_index()
)

print("\nEXTENDED OVER COUNT BY DELIVERY COUNT AND RECORDED WIDE/OVERSTEP")
display(
    df_extended_over_profile.groupby(
        ["delivery_count", "recorded_wide_or_overstep"]
    ).size().reset_index(name="over_count")
)

print("\nTOTAL EXTENDED OVERS:", len(df_extended_over_profile))
print(
    "WITH RECORDED WIDE/OVERSTEP:",
    df_extended_over_profile["recorded_wide_or_overstep"].sum()
)
print(
    "WITHOUT RECORDED WIDE/OVERSTEP:",
    (~df_extended_over_profile["recorded_wide_or_overstep"]).sum()
)

assert len(df_extended_over_profile) == 6450
assert (
    (~df_extended_over_profile["recorded_wide_or_overstep"]).sum()
    == 147
)

print("\nPASS: Extended-over event composition audited.")

EXTENDED OVER COMPOSITION SUMMARY


,recorded_wide_or_overstep,over_count,delivery_records,total_wides,total_oversteps,total_other,total_legbyes,total_byes,total_no_extras,total_extras2_nonzero
0,False,147,1036,0,0,153,14,7,862,9
1,True,6303,45785,7330,547,95,614,132,37067,25



EXTENDED OVER COUNT BY DELIVERY COUNT AND RECORDED WIDE/OVERSTEP


,delivery_count,recorded_wide_or_overstep,over_count
0,7,False,141
1,7,True,4987
2,8,False,5
3,8,True,1043
4,9,False,1
5,9,True,217
6,10,True,41
7,11,True,13
8,13,True,2



TOTAL EXTENDED OVERS: 6450
WITH RECORDED WIDE/OVERSTEP: 6303
WITHOUT RECORDED WIDE/OVERSTEP: 147

PASS: Extended-over event composition audited.


# Legal-ball reconstruction

### Delivery-level evidence classification

In [62]:
df_legal_ball_evidence = df_eligible_regulation_deliveries.copy()

extras_evidence_map = {
    1: "RECORDED_OVERSTEP",
    2: "RECORDED_WIDE",
    3: "RECORDED_LEGBYES",
    4: "RECORDED_BYES",
    5: "RECORDED_OTHER",
    6: "RECORDED_NO_EXTRAS",
    7: "RECORDED_PENALTY"
}

df_legal_ball_evidence["legal_ball_evidence_class"] = (
    df_legal_ball_evidence["extras1_runs"]
    .map(extras_evidence_map)
    .fillna("UNMAPPED_SOURCE_CODE")
)

# Summarize the source evidence without asserting legal-ball status
evidence_summary = (
    df_legal_ball_evidence
    .groupby("legal_ball_evidence_class", dropna=False)
    .agg(
        delivery_records=("match_id", "size"),
        matches=("match_id", "nunique"),
        innings=("innings", "nunique"),
        extras2_nonzero=("extras_2_runs", lambda x: (x != 0).sum()),
        total_runs=("runs", "sum")
    )
    .reset_index()
    .sort_values("legal_ball_evidence_class")
)

print("DELIVERY-LEVEL LEGAL-BALL EVIDENCE SUMMARY")
display(evidence_summary)

print("\nTOTAL REGULATION DELIVERY RECORDS:",
      len(df_legal_ball_evidence))

print("UNMAPPED SOURCE CODES:",
      (df_legal_ball_evidence["legal_ball_evidence_class"]
       == "UNMAPPED_SOURCE_CODE").sum())

assert len(df_legal_ball_evidence) == 192235
assert df_legal_ball_evidence["legal_ball_evidence_class"].notna().all()
assert (
    df_legal_ball_evidence["legal_ball_evidence_class"]
    == "UNMAPPED_SOURCE_CODE"
).sum() == 0

print("\nPASS: Every regulation delivery has a documented source-evidence class.")

DELIVERY-LEVEL LEGAL-BALL EVIDENCE SUMMARY


,legal_ball_evidence_class,delivery_records,matches,innings,extras2_nonzero,total_runs
0,RECORDED_BYES,505,366,2,0,0
1,RECORDED_LEGBYES,2635,772,2,0,0
2,RECORDED_NO_EXTRAS,180862,813,2,1,257555
3,RECORDED_OTHER,256,207,2,13,403
4,RECORDED_OVERSTEP,557,362,2,21,824
5,RECORDED_PENALTY,1,1,1,0,0
6,RECORDED_WIDE,7419,807,2,0,0



TOTAL REGULATION DELIVERY RECORDS: 192235
UNMAPPED SOURCE CODES: 0

PASS: Every regulation delivery has a documented source-evidence class.


### Conservative delivery status

In [63]:
non_legal_evidence = [
    "RECORDED_WIDE",
    "RECORDED_OVERSTEP"
]

unresolved_evidence = [
    "RECORDED_OTHER",
    "RECORDED_PENALTY"
]

df_legal_ball_evidence["legal_ball_evidence_status"] = "NO_RECORDED_WIDE_OR_OVERSTEP"

df_legal_ball_evidence.loc[
    df_legal_ball_evidence["legal_ball_evidence_class"].isin(non_legal_evidence),
    "legal_ball_evidence_status"
] = "RECORDED_NON_LEGAL_EVIDENCE"

df_legal_ball_evidence.loc[
    df_legal_ball_evidence["legal_ball_evidence_class"].isin(unresolved_evidence),
    "legal_ball_evidence_status"
] = "UNRESOLVED"

legal_ball_status_summary = (
    df_legal_ball_evidence
    .groupby("legal_ball_evidence_status")
    .agg(
        delivery_records=("match_id", "size"),
        matches=("match_id", "nunique"),
        total_runs=("runs", "sum"),
        extras2_nonzero=("extras_2_runs", lambda x: (x != 0).sum())
    )
    .reset_index()
)

print("CONSERVATIVE LEGAL-BALL EVIDENCE STATUS")
display(legal_ball_status_summary)

print("\nSOURCE CATEGORY CROSS-CHECK")
display(
    pd.crosstab(
        df_legal_ball_evidence["legal_ball_evidence_class"],
        df_legal_ball_evidence["legal_ball_evidence_status"]
    )
)

assert len(df_legal_ball_evidence) == 192235
assert df_legal_ball_evidence["legal_ball_evidence_status"].notna().all()

print("\nPASS: Every delivery has a conservative evidence status.")
print("NOTE: This cell does not calculate or assert a final legal-ball count.")

CONSERVATIVE LEGAL-BALL EVIDENCE STATUS


,legal_ball_evidence_status,delivery_records,matches,total_runs,extras2_nonzero
0,NO_RECORDED_WIDE_OR_OVERSTEP,184002,813,257555,1
1,RECORDED_NON_LEGAL_EVIDENCE,7976,810,824,21
2,UNRESOLVED,257,208,403,13



SOURCE CATEGORY CROSS-CHECK


legal_ball_evidence_status,NO_RECORDED_WIDE_OR_OVERSTEP,RECORDED_NON_LEGAL_EVIDENCE,UNRESOLVED
legal_ball_evidence_class,,,
RECORDED_BYES,505,0,0
RECORDED_LEGBYES,2635,0,0
RECORDED_NO_EXTRAS,180862,0,0
RECORDED_OTHER,0,0,256
RECORDED_OVERSTEP,0,557,0
RECORDED_PENALTY,0,0,1
RECORDED_WIDE,0,7419,0



PASS: Every delivery has a conservative evidence status.
NOTE: This cell does not calculate or assert a final legal-ball count.


### Unresolved delivery inspection

In [64]:
df_legal_ball_unresolved = df_legal_ball_evidence[
    df_legal_ball_evidence["legal_ball_evidence_status"] == "UNRESOLVED"
].copy()

unresolved_columns = [
    "match_id",
    "innings",
    "overid",
    "innings_ball_num",
    "over_ball_num",
    "striker_id",
    "non_striker_id",
    "bowler_id",
    "runs",
    "extras1_runs",
    "extras_type",
    "extras_2_runs",
    "is_wicket",
    "dismissal_type",
    "dismissed_player_id"
]

available_columns = [
    col for col in unresolved_columns
    if col in df_legal_ball_unresolved.columns
]

print("UNRESOLVED DELIVERY RECORDS:", len(df_legal_ball_unresolved))

print("\nUNRESOLVED SOURCE CATEGORY COUNTS")
display(
    df_legal_ball_unresolved["legal_ball_evidence_class"]
    .value_counts(dropna=False)
    .rename_axis("source_category")
    .reset_index(name="delivery_records")
)

print("\nUNRESOLVED RECORDS — DETAIL")
display(
    df_legal_ball_unresolved[available_columns]
    .sort_values(["match_id", "innings", "overid", "innings_ball_num"])
    .reset_index(drop=True)
)

print("\nUNRESOLVED RECORDS — FIELD PROFILE")
display(
    df_legal_ball_unresolved[available_columns]
    .describe(include="all")
    .T
)

assert len(df_legal_ball_unresolved) == 257
assert df_legal_ball_unresolved["legal_ball_evidence_class"].value_counts().to_dict() == {
    "RECORDED_OTHER": 256,
    "RECORDED_PENALTY": 1
}

print("\nPASS: All 257 unresolved records isolated for review.")

UNRESOLVED DELIVERY RECORDS: 257

UNRESOLVED SOURCE CATEGORY COUNTS


,source_category,delivery_records
0,RECORDED_OTHER,256
1,RECORDED_PENALTY,1



UNRESOLVED RECORDS — DETAIL


,match_id,innings,overid,innings_ball_num,over_ball_num,striker_id,non_striker_id,bowler_id,runs,extras1_runs,extras_type,extras_2_runs,is_wicket,dismissal_type,dismissed_player_id
0,13513,1,10,57,1,4986,3125,2304,0,5,other,0,False,NaN,NaN
1,13515,1,16,99,6,4986,494,4499,0,5,other,0,False,NaN,NaN
2,13515,2,19,119,5,1924,786,1772,2,5,other,0,False,NaN,NaN
3,13614,1,18,111,5,2436,6160,829,0,5,other,4,False,NaN,NaN
4,13614,2,2,7,1,2364,250,19312,0,5,other,0,False,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
252,54210,1,16,98,4,49027,4853,360,0,5,other,0,False,NaN,NaN
253,54210,2,16,102,6,3647,37052,4857,0,5,other,0,False,NaN,NaN
254,54213,1,16,96,2,4853,6512,8221,4,5,other,0,False,NaN,NaN
255,55078,1,20,124,4,39657,2399,71727,0,5,other,1,False,NaN,NaN



UNRESOLVED RECORDS — FIELD PROFILE


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
match_id,257.0,NaN,NaN,NaN,32324.241245,13040.153395,13513.0,20204.0,32136.0,42784.0,55561.0
innings,257.0,NaN,NaN,NaN,1.470817,0.500122,1.0,1.0,1.0,2.0,2.0
overid,257.0,NaN,NaN,NaN,13.782101,5.696009,1.0,9.0,16.0,18.0,20.0
innings_ball_num,257.0,NaN,NaN,NaN,83.844358,35.95243,1.0,54.0,98.0,113.0,129.0
over_ball_num,257.0,NaN,NaN,NaN,3.859922,1.957515,1.0,2.0,4.0,5.0,9.0
striker_id,257.0,NaN,NaN,NaN,6544.564202,9008.642552,8.0,2490.0,4409.0,6396.0,56817.0
non_striker_id,257.0,NaN,NaN,NaN,6577.844358,8771.022514,8.0,2258.0,4409.0,6512.0,56830.0
bowler_id,257.0,NaN,NaN,NaN,11145.618677,13998.660673,28.0,3623.0,5809.0,8530.0,71727.0
runs,257.0,NaN,NaN,NaN,1.568093,2.047348,0.0,0.0,1.0,4.0,6.0
extras1_runs,257.0,NaN,NaN,NaN,5.007782,0.124757,5.0,5.0,5.0,5.0,7.0



PASS: All 257 unresolved records isolated for review.


### Finalize the legal-ball evidence register

In [65]:
legal_ball_treatment_map = {
    "RECORDED_WIDE": "NON_LEGAL_EVIDENCE",
    "RECORDED_OVERSTEP": "NON_LEGAL_EVIDENCE",
    "RECORDED_OTHER": "UNRESOLVED",
    "RECORDED_PENALTY": "UNRESOLVED",
    "RECORDED_BYES": "NO_RECORDED_WIDE_OR_OVERSTEP",
    "RECORDED_LEGBYES": "NO_RECORDED_WIDE_OR_OVERSTEP",
    "RECORDED_NO_EXTRAS": "NO_RECORDED_WIDE_OR_OVERSTEP"
}

df_legal_ball_evidence["legal_ball_final_treatment"] = (
    df_legal_ball_evidence["legal_ball_evidence_class"]
    .map(legal_ball_treatment_map)
)

assert df_legal_ball_evidence["legal_ball_final_treatment"].notna().all()
assert len(df_legal_ball_evidence) == 192235

legal_ball_final_summary = (
    df_legal_ball_evidence
    .groupby(
        ["legal_ball_evidence_class", "legal_ball_final_treatment"],
        dropna=False
    )
    .size()
    .reset_index(name="delivery_records")
    .sort_values("legal_ball_evidence_class")
)

print("FINAL LEGAL-BALL EVIDENCE TREATMENT")
display(legal_ball_final_summary)

print("\nTREATMENT TOTALS")
display(
    df_legal_ball_evidence["legal_ball_final_treatment"]
    .value_counts()
    .rename_axis("treatment")
    .reset_index(name="delivery_records")
)

print("\nUNRESOLVED RECORDS:",
      (df_legal_ball_evidence["legal_ball_final_treatment"] == "UNRESOLVED").sum())

print("EXPLICIT NON-LEGAL EVIDENCE:",
      (df_legal_ball_evidence["legal_ball_final_treatment"] == "NON_LEGAL_EVIDENCE").sum())

print("\nPASS: Evidence-based legal-ball treatment finalized without imputing legal status.")

FINAL LEGAL-BALL EVIDENCE TREATMENT


,legal_ball_evidence_class,legal_ball_final_treatment,delivery_records
0,RECORDED_BYES,NO_RECORDED_WIDE_OR_OVERSTEP,505
1,RECORDED_LEGBYES,NO_RECORDED_WIDE_OR_OVERSTEP,2635
2,RECORDED_NO_EXTRAS,NO_RECORDED_WIDE_OR_OVERSTEP,180862
3,RECORDED_OTHER,UNRESOLVED,256
4,RECORDED_OVERSTEP,NON_LEGAL_EVIDENCE,557
5,RECORDED_PENALTY,UNRESOLVED,1
6,RECORDED_WIDE,NON_LEGAL_EVIDENCE,7419



TREATMENT TOTALS


,treatment,delivery_records
0,NO_RECORDED_WIDE_OR_OVERSTEP,184002
1,NON_LEGAL_EVIDENCE,7976
2,UNRESOLVED,257



UNRESOLVED RECORDS: 257
EXPLICIT NON-LEGAL EVIDENCE: 7976

PASS: Evidence-based legal-ball treatment finalized without imputing legal status.


### Add the legal-ball exception

In [66]:
legal_ball_exception = {
    "exception_id": "P2-LEGAL-001",
    "exception_type": "LEGAL_BALL_RECONSTRUCTION",
    "dataset": "ball_by_ball",
    "match_id": None,
    "affected_records": 257,
    "field": "extras1_runs / extras_type / extras_2_runs",
    "observation": (
        "256 deliveries are coded as other and 1 as Penalty. "
        "Available source fields do not establish their legal-ball treatment."
    ),
    "treatment": (
        "Retain all 257 records as UNRESOLVED. "
        "Do not infer legal-ball status or impute classifications. "
        "Preserve explicit wide and overstep evidence separately."
    ),
    "status": "Documented - Unresolved",
    "innings": None,
    "overid": None,
    "innings_ball_num": None,
    "bowler_id": None,
    "extras1_runs": None,
    "extras_type": "other (256); Penalty (1)",
    "extras_2_runs": None,
    "runs": None
}

assert "P2-LEGAL-001" not in set(phase2_exceptions["exception_id"])

phase2_exceptions = pd.concat(
    [
        phase2_exceptions,
        pd.DataFrame([legal_ball_exception])
    ],
    ignore_index=True
)

assert phase2_exceptions["exception_id"].is_unique
assert "P2-LEGAL-001" in set(phase2_exceptions["exception_id"])

print("UPDATED EXCEPTION REGISTER")
print("Total exceptions:", len(phase2_exceptions))
display(
    phase2_exceptions[
        phase2_exceptions["exception_id"] == "P2-LEGAL-001"
    ]
)

print("\nEXCEPTION COUNTS BY TYPE")
display(
    phase2_exceptions["exception_type"]
    .value_counts()
    .rename_axis("exception_type")
    .reset_index(name="exceptions")
)

print("\nPASS: Legal-ball limitation added to the exception register.")

UPDATED EXCEPTION REGISTER
Total exceptions: 24


/tmp/ipykernel_21825/3120675525.py:30: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  phase2_exceptions = pd.concat(


,exception_id,exception_type,dataset,match_id,affected_records,field,observation,treatment,status,innings,overid,innings_ball_num,bowler_id,extras1_runs,extras_type,extras_2_runs,runs
23,P2-LEGAL-001,LEGAL_BALL_RECONSTRUCTION,ball_by_ball,None,257,extras1_runs / extras_type / extras_2_runs,256 deliveries are coded as other and 1 as Pen...,Retain all 257 records as UNRESOLVED. Do not i...,Documented - Unresolved,NaN,NaN,NaN,NaN,NaN,other (256); Penalty (1),NaN,NaN



EXCEPTION COUNTS BY TYPE


,exception_type,exceptions
0,NON_WICKET_WITH_DISMISSED_PLAYER_ID,11
1,NONZERO_EXTRAS2_WITH_RECORDED_OTHER,9
2,MISSING_PLAYED_BATTING_ORDER,1
3,WICKET_MISSING_DISMISSED_PLAYER_ID,1
4,DISMISSAL_LABEL_SPELLING_VARIANT,1
5,LEGAL_BALL_RECONSTRUCTION,1



PASS: Legal-ball limitation added to the exception register.


### Save the updated Phase 2 audit artifacts

In [67]:
from pathlib import Path

output_dir = Path("/content/phase2_outputs")
output_dir.mkdir(parents=True, exist_ok=True)

audit_files = {
    "phase2_exception_register.csv": phase2_exceptions,
    "phase2_legal_ball_evidence.csv": df_legal_ball_evidence,
    "phase2_unexplained_extended_over_register.csv": df_unexplained_over_full,
    "phase2_dismissal_classified.csv": df_dismissal_classified,
    "phase2_team_identity_audit.csv": df_team_identity_audit,
    "phase2_team_identity_exceptions.csv": df_team_identity_exceptions,
    "phase2_over_sequence_audit.csv": df_over_sequence
}

for filename, dataframe in audit_files.items():
    output_path = output_dir / filename
    dataframe.to_csv(output_path, index=False)
    assert output_path.exists() and output_path.stat().st_size > 0
    print(f"Saved: {filename} | Rows: {len(dataframe):,} | "
          f"Size: {output_path.stat().st_size:,} bytes")

print("\nTOTAL AUDIT FILES SAVED:", len(audit_files))
print("OUTPUT DIRECTORY:", output_dir)

Saved: phase2_exception_register.csv | Rows: 24 | Size: 6,829 bytes
Saved: phase2_legal_ball_evidence.csv | Rows: 192,235 | Size: 53,412,266 bytes
Saved: phase2_unexplained_extended_over_register.csv | Rows: 147 | Size: 4,500 bytes
Saved: phase2_dismissal_classified.csv | Rows: 192,261 | Size: 39,264,086 bytes
Saved: phase2_team_identity_audit.csv | Rows: 4 | Size: 295 bytes
Saved: phase2_team_identity_exceptions.csv | Rows: 0 | Size: 1 bytes
Saved: phase2_over_sequence_audit.csv | Rows: 30,921 | Size: 919,553 bytes

TOTAL AUDIT FILES SAVED: 7
OUTPUT DIRECTORY: /content/phase2_outputs


#### Assemble the integrated delivery dataset

In [68]:
# Start with the approved regulation-delivery population
df_phase2_deliveries = df_eligible_regulation_deliveries.copy()

# Attach normalized dismissal labels
dismissal_columns = [
    "match_id",
    "innings",
    "innings_ball_num",
    "dismissal_type_original",
    "dismissal_type_normalized"
]

df_phase2_deliveries = df_phase2_deliveries.merge(
    df_dismissal_classified[dismissal_columns],
    on=["match_id", "innings", "innings_ball_num"],
    how="left",
    validate="one_to_one"
)

# Attach legal-ball evidence classifications
legal_ball_columns = [
    "match_id",
    "innings",
    "innings_ball_num",
    "legal_ball_evidence_class",
    "legal_ball_evidence_status",
    "legal_ball_final_treatment"
]

df_phase2_deliveries = df_phase2_deliveries.merge(
    df_legal_ball_evidence[legal_ball_columns],
    on=["match_id", "innings", "innings_ball_num"],
    how="left",
    validate="one_to_one"
)

# Validate the integrated population
assert len(df_phase2_deliveries) == 192235
assert df_phase2_deliveries[
    ["legal_ball_evidence_class",
     "legal_ball_evidence_status",
     "legal_ball_final_treatment"]
].notna().all().all()

assert df_phase2_deliveries["match_id"].nunique() == 813

print("INTEGRATED DATASET SHAPE:", df_phase2_deliveries.shape)
print("UNIQUE MATCHES:", df_phase2_deliveries["match_id"].nunique())
print("REGULATION DELIVERIES:", len(df_phase2_deliveries))

print("\nLEGAL-BALL TREATMENT COUNTS")
display(
    df_phase2_deliveries["legal_ball_final_treatment"]
    .value_counts()
    .rename_axis("treatment")
    .reset_index(name="delivery_records")
)

print("\nNORMALIZED DISMISSAL LABEL COUNTS")
display(
    df_phase2_deliveries["dismissal_type_normalized"]
    .value_counts(dropna=False)
    .rename_axis("dismissal_type")
    .reset_index(name="delivery_records")
)

print("\nPASS: Integrated regulation-delivery dataset assembled.")

INTEGRATED DATASET SHAPE: (192235, 38)
UNIQUE MATCHES: 813
REGULATION DELIVERIES: 192235

LEGAL-BALL TREATMENT COUNTS


,treatment,delivery_records
0,NO_RECORDED_WIDE_OR_OVERSTEP,184002
1,NON_LEGAL_EVIDENCE,7976
2,UNRESOLVED,257



NORMALIZED DISMISSAL LABEL COUNTS


,dismissal_type,delivery_records
0,NaN,181843
1,Caught,7264
2,Bowled,1675
3,LBW,663
4,RunOut,567
5,Stump,198
6,Hit Wicket,17
7,Retire Out,6
8,Obstructing the field,2



PASS: Integrated regulation-delivery dataset assembled.


### Reconcile dismissal counts

In [69]:
# Full eligible population includes regulation + Super Over deliveries
full_dismissal_counts = (
    df_dismissal_classified["dismissal_type_normalized"]
    .value_counts(dropna=False)
    .rename("full_eligible_count")
)

# Regulation-only integrated population
regulation_dismissal_counts = (
    df_phase2_deliveries["dismissal_type_normalized"]
    .value_counts(dropna=False)
    .rename("regulation_count")
)

dismissal_reconciliation = pd.concat(
    [full_dismissal_counts, regulation_dismissal_counts],
    axis=1
).fillna(0).astype(int)

dismissal_reconciliation["difference"] = (
    dismissal_reconciliation["full_eligible_count"]
    - dismissal_reconciliation["regulation_count"]
)

print("DISMISSAL COUNT RECONCILIATION")
display(dismissal_reconciliation.reset_index(names="dismissal_type"))

print("\nSUPER OVER DELIVERY COUNT:",
      len(df_eligible_super_over_deliveries))

print("FULL ELIGIBLE DELIVERY COUNT:",
      len(df_dismissal_classified))

print("REGULATION DELIVERY COUNT:",
      len(df_phase2_deliveries))

assert len(df_dismissal_classified) - len(df_phase2_deliveries) == 26
assert dismissal_reconciliation["difference"].sum() == 26

print("\nPASS: Dismissal-count differences reconcile to the 26 Super Over deliveries.")

DISMISSAL COUNT RECONCILIATION


,dismissal_type,full_eligible_count,regulation_count,difference
0,NaN,181861,181843,18
1,Caught,7269,7264,5
2,Bowled,1676,1675,1
3,LBW,663,663,0
4,RunOut,569,567,2
5,Stump,198,198,0
6,Hit Wicket,17,17,0
7,Retire Out,6,6,0
8,Obstructing the field,2,2,0



SUPER OVER DELIVERY COUNT: 26
FULL ELIGIBLE DELIVERY COUNT: 192261
REGULATION DELIVERY COUNT: 192235

PASS: Dismissal-count differences reconcile to the 26 Super Over deliveries.


### Inspect join keys and context schemas

In [71]:
# PHASE 2 — INTEGRATED DATASET
# CELL 9: VALIDATE CONTEXT JOIN KEYS

print("DELIVERY DATASET — RELEVANT COLUMNS")
display([
    col for col in df_phase2_deliveries.columns
    if any(term in col.lower() for term in
           ["match", "comp", "team", "innings", "striker", "bowler"])
])

print("\nMATCH TABLE COLUMNS")
display(df_matches_raw.columns.tolist())

print("\nCOMPETITION TABLE COLUMNS")
display(df_competitions_raw.columns.tolist())

print("\nTEAM TABLE COLUMNS")
display(df_teams_raw.columns.tolist())

print("\nKEY UNIQUENESS CHECKS")

print("Eligible match IDs unique:",
      df_eligible_matches["match_id"].is_unique)

print("Raw match IDs unique:",
      df_matches_raw["match_id"].is_unique)

print("Competition IDs unique:",
      df_competitions_raw["comp_id"].is_unique)

print("Team IDs globally unique:",
      df_teams_raw["team_id"].is_unique)

print("Competition-team composite keys unique:",
      not df_teams_raw.duplicated(
          subset=["comp_id", "team_id"]
      ).any())

assert df_eligible_matches["match_id"].is_unique
assert df_competitions_raw["comp_id"].is_unique

print("\nPASS: Match and competition keys validated.")
print("Team key uniqueness requires composite-key verification.")

DELIVERY DATASET — RELEVANT COLUMNS


['match_id',
 'innings',
 'innings_ball_num',
 'batting_team_id',
 'bowling_team_id',
 'striker_id',
 'striker_name',
 'non_striker_id',
 'non_striker_name',
 'bowler_id',
 'bowler_name']


MATCH TABLE COLUMNS


['match_id',
 'comp_id',
 'match_number',
 'match_date',
 'stage',
 'venue_id',
 'venue_country',
 'home_team_id',
 'home_team',
 'away_team_id',
 'away_team',
 'home_team_captain_id',
 'away_team_captain_id',
 'toss_win_team_id',
 'toss_opted',
 'man_of_the_match',
 'first_inn_revised_overs',
 'second_inn_revised_overs',
 'revised_target',
 'knock_out',
 'win_team_id',
 'result']


COMPETITION TABLE COLUMNS


['comp_id', 'comp_name', 'match_format', 'season']


TEAM TABLE COLUMNS


['comp_id', 'team_id', 'team_name']


KEY UNIQUENESS CHECKS
Eligible match IDs unique: True
Raw match IDs unique: True
Competition IDs unique: True
Team IDs globally unique: False
Competition-team composite keys unique: True

PASS: Match and competition keys validated.
Team key uniqueness requires composite-key verification.


#### Validate team joins against the eligible matches

In [72]:
# Add competition ID to eligible match team references
match_team_references = pd.concat([
    df_eligible_matches[
        ["match_id", "comp_id", "home_team_id"]
    ].rename(columns={"home_team_id": "team_id"}).assign(
        team_role="home"
    ),
    df_eligible_matches[
        ["match_id", "comp_id", "away_team_id"]
    ].rename(columns={"away_team_id": "team_id"}).assign(
        team_role="away"
    )
], ignore_index=True)

# Check against the composite team key
team_key_lookup = df_teams_raw[
    ["comp_id", "team_id", "team_name"]
].drop_duplicates()

team_reference_audit = match_team_references.merge(
    team_key_lookup,
    on=["comp_id", "team_id"],
    how="left",
    indicator=True,
    validate="many_to_one"
)

unmatched_team_references = team_reference_audit[
    team_reference_audit["_merge"] == "left_only"
].copy()

print("TEAM REFERENCES CHECKED:", len(team_reference_audit))
print("EXPECTED TEAM REFERENCES:", len(df_eligible_matches) * 2)
print("MATCHED TEAM REFERENCES:",
      (team_reference_audit["_merge"] == "both").sum())
print("UNMATCHED TEAM REFERENCES:", len(unmatched_team_references))

if len(unmatched_team_references) > 0:
    display(unmatched_team_references)

assert len(team_reference_audit) == len(df_eligible_matches) * 2

print("\nPASS: All eligible match team references have been checked.")

TEAM REFERENCES CHECKED: 1626
EXPECTED TEAM REFERENCES: 1626
MATCHED TEAM REFERENCES: 1626
UNMATCHED TEAM REFERENCES: 0

PASS: All eligible match team references have been checked.


### Attach match, competition, and team context

In [73]:
# 1. Attach match context
match_context_columns = [
    "match_id",
    "comp_id",
    "match_number",
    "match_date",
    "stage",
    "venue_id",
    "venue_country",
    "home_team_id",
    "home_team",
    "away_team_id",
    "away_team",
    "toss_win_team_id",
    "toss_opted",
    "win_team_id",
    "result",
    "revised_target",
    "first_inn_revised_overs",
    "second_inn_revised_overs",
    "knock_out"
]

df_phase2_deliveries = df_phase2_deliveries.merge(
    df_eligible_matches[match_context_columns],
    on="match_id",
    how="left",
    validate="many_to_one"
)

assert len(df_phase2_deliveries) == 192235
assert df_phase2_deliveries["comp_id"].notna().all()

# 2. Attach competition context
df_phase2_deliveries = df_phase2_deliveries.merge(
    df_competitions_raw[
        ["comp_id", "comp_name", "match_format", "season"]
    ],
    on="comp_id",
    how="left",
    validate="many_to_one"
)

assert len(df_phase2_deliveries) == 192235
assert df_phase2_deliveries["comp_name"].notna().all()

# 3. Attach batting team name using competition + team ID
batting_team_lookup = (
    df_teams_raw[
        ["comp_id", "team_id", "team_name"]
    ]
    .rename(columns={
        "team_id": "batting_team_id",
        "team_name": "batting_team_name"
    })
)

df_phase2_deliveries = df_phase2_deliveries.merge(
    batting_team_lookup,
    on=["comp_id", "batting_team_id"],
    how="left",
    validate="many_to_one"
)

# 4. Attach bowling team name using competition + team ID
bowling_team_lookup = (
    df_teams_raw[
        ["comp_id", "team_id", "team_name"]
    ]
    .rename(columns={
        "team_id": "bowling_team_id",
        "team_name": "bowling_team_name"
    })
)

df_phase2_deliveries = df_phase2_deliveries.merge(
    bowling_team_lookup,
    on=["comp_id", "bowling_team_id"],
    how="left",
    validate="many_to_one"
)

# 5. Final checks
assert len(df_phase2_deliveries) == 192235
assert df_phase2_deliveries["match_id"].nunique() == 813
assert df_phase2_deliveries["comp_name"].notna().all()
assert df_phase2_deliveries["batting_team_name"].notna().all()
assert df_phase2_deliveries["bowling_team_name"].notna().all()

print("INTEGRATED DATASET SHAPE:", df_phase2_deliveries.shape)
print("UNIQUE MATCHES:", df_phase2_deliveries["match_id"].nunique())
print("UNIQUE COMPETITIONS:", df_phase2_deliveries["comp_id"].nunique())

print("\nMISSING CONTEXT VALUES")
display(
    df_phase2_deliveries[
        ["comp_name", "match_format", "season",
         "batting_team_name", "bowling_team_name"]
    ].isna().sum().rename("missing_values").to_frame()
)

print("\nSAMPLE INTEGRATED RECORDS")
display(
    df_phase2_deliveries[
        ["match_id", "match_date", "comp_name", "match_format",
         "innings", "batting_team_id", "batting_team_name",
         "bowling_team_id", "bowling_team_name",
         "legal_ball_final_treatment"]
    ].head(10)
)

print("\nPASS: Match, competition and team context attached successfully.")

INTEGRATED DATASET SHAPE: (192235, 61)
UNIQUE MATCHES: 813
UNIQUE COMPETITIONS: 136

MISSING CONTEXT VALUES


,missing_values
comp_name,0
match_format,0
season,0
batting_team_name,0
bowling_team_name,0



SAMPLE INTEGRATED RECORDS


,match_id,match_date,comp_name,match_format,innings,batting_team_id,batting_team_name,bowling_team_id,bowling_team_name,legal_ball_final_treatment
0,13511,2022-01-22,England in West Indies T20I Series 2022,T20I,1,22,England,28,West Indies,NO_RECORDED_WIDE_OR_OVERSTEP
1,13511,2022-01-22,England in West Indies T20I Series 2022,T20I,1,22,England,28,West Indies,NO_RECORDED_WIDE_OR_OVERSTEP
2,13511,2022-01-22,England in West Indies T20I Series 2022,T20I,1,22,England,28,West Indies,NO_RECORDED_WIDE_OR_OVERSTEP
3,13511,2022-01-22,England in West Indies T20I Series 2022,T20I,1,22,England,28,West Indies,NO_RECORDED_WIDE_OR_OVERSTEP
4,13511,2022-01-22,England in West Indies T20I Series 2022,T20I,1,22,England,28,West Indies,NO_RECORDED_WIDE_OR_OVERSTEP
5,13511,2022-01-22,England in West Indies T20I Series 2022,T20I,1,22,England,28,West Indies,NO_RECORDED_WIDE_OR_OVERSTEP
6,13511,2022-01-22,England in West Indies T20I Series 2022,T20I,1,22,England,28,West Indies,NO_RECORDED_WIDE_OR_OVERSTEP
7,13511,2022-01-22,England in West Indies T20I Series 2022,T20I,1,22,England,28,West Indies,NO_RECORDED_WIDE_OR_OVERSTEP
8,13511,2022-01-22,England in West Indies T20I Series 2022,T20I,1,22,England,28,West Indies,NO_RECORDED_WIDE_OR_OVERSTEP
9,13511,2022-01-22,England in West Indies T20I Series 2022,T20I,1,22,England,28,West Indies,NO_RECORDED_WIDE_OR_OVERSTEP



PASS: Match, competition and team context attached successfully.


#### Integrated dataset integrity audit

In [75]:
# FIX: MISSING VALUE PROFILE

missing_profile = (
    df_phase2_deliveries.isna()
    .sum()
    .rename("missing_values")
    .reset_index()
)

missing_profile.columns = ["column", "missing_values"]

display(
    missing_profile[
        missing_profile["missing_values"] > 0
    ]
)

print("Columns with missing values:",
      (missing_profile["missing_values"] > 0).sum())

print("Total missing values:",
      missing_profile["missing_values"].sum())

,column,missing_values
20,dismissal_type_id,181843
21,dismissal_type,181843
22,dismissed_player_id,181833
23,extras_type,180862
28,stroke,5960
29,delivery_style,376
30,over_or_around,396
33,dismissal_type_original,181843
34,dismissal_type_normalized,181843
50,win_team_id,179


Columns with missing values: 13
Total missing values: 1664273


#### Verify missingness patterns

In [76]:
print("1. DISMISSAL FIELD CONSISTENCY")

print("Non-wicket deliveries:",
      (~df_phase2_deliveries["is_wicket"]).sum())

print("Missing dismissal types:",
      df_phase2_deliveries["dismissal_type_normalized"].isna().sum())

print("Wicket deliveries:",
      df_phase2_deliveries["is_wicket"].sum())

print("Wickets with missing dismissal type:",
      df_phase2_deliveries.loc[
          df_phase2_deliveries["is_wicket"],
          "dismissal_type_normalized"
      ].isna().sum())

print("\n2. EXTRAS FIELD CONSISTENCY")

display(
    df_phase2_deliveries.groupby(
        "legal_ball_evidence_class",
        dropna=False
    ).agg(
        records=("match_id", "size"),
        missing_extras_type=("extras_type", lambda x: x.isna().sum())
    ).reset_index()
)

print("\n3. MISSING WIN TEAM ID")

missing_win_team = df_phase2_deliveries[
    df_phase2_deliveries["win_team_id"].isna()
]

print("Affected delivery records:", len(missing_win_team))
print("Affected matches:", missing_win_team["match_id"].nunique())

display(
    missing_win_team[
        ["match_id", "comp_name", "match_date", "result",
         "home_team", "away_team", "win_team_id"]
    ].drop_duplicates().head(30)
)

print("\n4. REVISED MATCH METADATA")

revised_metadata = [
    "revised_target",
    "first_inn_revised_overs",
    "second_inn_revised_overs"
]

display(
    df_phase2_deliveries.groupby("match_id")[revised_metadata]
    .first()
    .isna()
    .sum()
    .rename("matches_with_missing_values")
    .to_frame()
)

print("\nPASS: Missingness patterns inspected; no values imputed.")

1. DISMISSAL FIELD CONSISTENCY
Non-wicket deliveries: 181843
Missing dismissal types: 181843
Wicket deliveries: 10392
Wickets with missing dismissal type: 0

2. EXTRAS FIELD CONSISTENCY


,legal_ball_evidence_class,records,missing_extras_type
0,RECORDED_BYES,505,0
1,RECORDED_LEGBYES,2635,0
2,RECORDED_NO_EXTRAS,180862,180862
3,RECORDED_OTHER,256,0
4,RECORDED_OVERSTEP,557,0
5,RECORDED_PENALTY,1,0
6,RECORDED_WIDE,7419,0



3. MISSING WIN TEAM ID
Affected delivery records: 179
Affected matches: 1


,match_id,comp_name,match_date,result,home_team,away_team,win_team_id
42082,18532,India in New Zealand T20I Series 2022/23,2022-11-22,Match Tied,New Zealand,India,NaN



4. REVISED MATCH METADATA


,matches_with_missing_values
revised_target,791
first_inn_revised_overs,798
second_inn_revised_overs,791



PASS: Missingness patterns inspected; no values imputed.


#### Validate the tied-match exception

In [77]:
match_outcome_audit = (
    df_phase2_deliveries[
        ["match_id", "result", "win_team_id"]
    ]
    .drop_duplicates()
)

missing_winner_matches = match_outcome_audit[
    match_outcome_audit["win_team_id"].isna()
].copy()

print("MATCHES WITH MISSING WIN_TEAM_ID:",
      len(missing_winner_matches))

display(missing_winner_matches)

print("\nRESULT DISTRIBUTION FOR MISSING WINNER MATCHES")
display(
    missing_winner_matches["result"]
    .value_counts(dropna=False)
    .rename_axis("result")
    .reset_index(name="matches")
)

assert len(missing_winner_matches) == 1
assert missing_winner_matches.iloc[0]["match_id"] == 18532
assert missing_winner_matches.iloc[0]["result"] == "Match Tied"

print("\nPASS: Missing winner ID is consistent with the recorded tied result.")

MATCHES WITH MISSING WIN_TEAM_ID: 1


,match_id,result,win_team_id
42082,18532,Match Tied,NaN



RESULT DISTRIBUTION FOR MISSING WINNER MATCHES


,result,matches
0,Match Tied,1



PASS: Missing winner ID is consistent with the recorded tied result.


### Final cross-table consistency checks

In [78]:
# 1. Match-to-competition consistency
match_comp_check = df_phase2_deliveries.merge(
    df_eligible_matches[["match_id", "comp_id"]],
    on="match_id",
    how="left",
    suffixes=("_integrated", "_source"),
    validate="many_to_one"
)

match_comp_mismatches = (
    match_comp_check["comp_id_integrated"]
    != match_comp_check["comp_id_source"]
).sum()

# 2. Batting team identity consistency
batting_check = df_phase2_deliveries.merge(
    df_teams_raw[["comp_id", "team_id", "team_name"]],
    left_on=["comp_id", "batting_team_id"],
    right_on=["comp_id", "team_id"],
    how="left",
    suffixes=("", "_source"),
    validate="many_to_one"
)

batting_name_mismatches = (
    batting_check["batting_team_name"]
    != batting_check["team_name"]
).sum()

# 3. Bowling team identity consistency
bowling_check = df_phase2_deliveries.merge(
    df_teams_raw[["comp_id", "team_id", "team_name"]],
    left_on=["comp_id", "bowling_team_id"],
    right_on=["comp_id", "team_id"],
    how="left",
    suffixes=("", "_source"),
    validate="many_to_one"
)

bowling_name_mismatches = (
    bowling_check["bowling_team_name"]
    != bowling_check["team_name"]
).sum()

# 4. Population boundary
integrated_match_ids = set(df_phase2_deliveries["match_id"].unique())
approved_match_ids = set(df_eligible_matches["match_id"].unique())

missing_match_ids = approved_match_ids - integrated_match_ids
unexpected_match_ids = integrated_match_ids - approved_match_ids

# 5. Report
print("MATCH-COMPETITION MISMATCHES:", match_comp_mismatches)
print("BATTING TEAM NAME MISMATCHES:", batting_name_mismatches)
print("BOWLING TEAM NAME MISMATCHES:", bowling_name_mismatches)
print("APPROVED MATCHES MISSING:", len(missing_match_ids))
print("UNEXPECTED MATCHES:", len(unexpected_match_ids))

print("\nFINAL POPULATION")
print("Delivery records:", len(df_phase2_deliveries))
print("Unique matches:", len(integrated_match_ids))
print("Unique competitions:", df_phase2_deliveries["comp_id"].nunique())

assert match_comp_mismatches == 0
assert batting_name_mismatches == 0
assert bowling_name_mismatches == 0
assert len(missing_match_ids) == 0
assert len(unexpected_match_ids) == 0
assert len(df_phase2_deliveries) == 192235

print("\nPASS: Cross-table consistency checks passed.")

MATCH-COMPETITION MISMATCHES: 0
BATTING TEAM NAME MISMATCHES: 0
BOWLING TEAM NAME MISMATCHES: 0
APPROVED MATCHES MISSING: 0
UNEXPECTED MATCHES: 0

FINAL POPULATION
Delivery records: 192235
Unique matches: 813
Unique competitions: 136

PASS: Cross-table consistency checks passed.


# Export the final Phase 2 dataset

In [79]:
from pathlib import Path

output_dir = Path("/content/phase2_outputs")
output_dir.mkdir(parents=True, exist_ok=True)

# 1. Save the final integrated regulation-delivery dataset
final_csv_path = output_dir / "phase2_integrated_cleaned_deliveries.csv"

df_phase2_deliveries.to_csv(final_csv_path, index=False)

assert final_csv_path.exists()
assert final_csv_path.stat().st_size > 0

print("FINAL DATASET CSV SAVED")
print("Path:", final_csv_path)
print("Rows:", len(df_phase2_deliveries))
print("Columns:", len(df_phase2_deliveries.columns))
print("Size (MB):", round(final_csv_path.stat().st_size / (1024 ** 2), 2))

# 2. Save a Parquet copy if a compatible engine is available
parquet_path = output_dir / "phase2_integrated_cleaned_deliveries.parquet"

try:
    df_phase2_deliveries.to_parquet(parquet_path, index=False)
    print("\nPARQUET SAVED:", parquet_path)
    print("Parquet size (MB):",
          round(parquet_path.stat().st_size / (1024 ** 2), 2))
except (ImportError, ModuleNotFoundError) as error:
    print("\nPARQUET NOT SAVED — compatible engine unavailable.")
    print("Reason:", str(error))

# 3. Refresh all audit artifacts
audit_files = {
    "phase2_exception_register.csv": phase2_exceptions,
    "phase2_legal_ball_evidence.csv": df_legal_ball_evidence,
    "phase2_unexplained_extended_over_register.csv": df_unexplained_over_full,
    "phase2_dismissal_classified.csv": df_dismissal_classified,
    "phase2_team_identity_audit.csv": df_team_identity_audit,
    "phase2_team_identity_exceptions.csv": df_team_identity_exceptions,
    "phase2_over_sequence_audit.csv": df_over_sequence
}

print("\nREFRESHING AUDIT FILES")

for filename, dataframe in audit_files.items():
    path = output_dir / filename
    dataframe.to_csv(path, index=False)

    assert path.exists()
    assert path.stat().st_size > 0

    print(f"{filename} | {len(dataframe):,} rows | "
          f"{path.stat().st_size:,} bytes")

# 4. Export manifest
export_manifest = pd.DataFrame([
    {
        "artifact": "phase2_integrated_cleaned_deliveries.csv",
        "rows": len(df_phase2_deliveries),
        "columns": len(df_phase2_deliveries.columns),
        "description": "Final integrated regulation-delivery dataset"
    }
] + [
    {
        "artifact": filename,
        "rows": len(dataframe),
        "columns": len(dataframe.columns),
        "description": "Phase 2 audit artifact"
    }
    for filename, dataframe in audit_files.items()
])

manifest_path = output_dir / "phase2_export_manifest.csv"
export_manifest.to_csv(manifest_path, index=False)

print("\nEXPORT MANIFEST")
display(export_manifest)

print("\nPASS: Final dataset and audit artifacts exported.")
print("OUTPUT DIRECTORY:", output_dir)

FINAL DATASET CSV SAVED
Path: /content/phase2_outputs/phase2_integrated_cleaned_deliveries.csv
Rows: 192235
Columns: 61
Size (MB): 85.96

PARQUET SAVED: /content/phase2_outputs/phase2_integrated_cleaned_deliveries.parquet
Parquet size (MB): 3.3

REFRESHING AUDIT FILES
phase2_exception_register.csv | 24 rows | 6,829 bytes
phase2_legal_ball_evidence.csv | 192,235 rows | 53,412,266 bytes
phase2_unexplained_extended_over_register.csv | 147 rows | 4,500 bytes
phase2_dismissal_classified.csv | 192,261 rows | 39,264,086 bytes
phase2_team_identity_audit.csv | 4 rows | 295 bytes
phase2_team_identity_exceptions.csv | 0 rows | 1 bytes
phase2_over_sequence_audit.csv | 30,921 rows | 919,553 bytes

EXPORT MANIFEST


,artifact,rows,columns,description
0,phase2_integrated_cleaned_deliveries.csv,192235,61,Final integrated regulation-delivery dataset
1,phase2_exception_register.csv,24,17,Phase 2 audit artifact
2,phase2_legal_ball_evidence.csv,192235,36,Phase 2 audit artifact
3,phase2_unexplained_extended_over_register.csv,147,12,Phase 2 audit artifact
4,phase2_dismissal_classified.csv,192261,35,Phase 2 audit artifact
5,phase2_team_identity_audit.csv,4,6,Phase 2 audit artifact
6,phase2_team_identity_exceptions.csv,0,0,Phase 2 audit artifact
7,phase2_over_sequence_audit.csv,30921,10,Phase 2 audit artifact



PASS: Final dataset and audit artifacts exported.
OUTPUT DIRECTORY: /content/phase2_outputs


### Verify exported files

In [80]:
from pathlib import Path
import pandas as pd

output_dir = Path("/content/phase2_outputs")

csv_path = output_dir / "phase2_integrated_cleaned_deliveries.csv"
parquet_path = output_dir / "phase2_integrated_cleaned_deliveries.parquet"
manifest_path = output_dir / "phase2_export_manifest.csv"

# 1. Verify files exist
for path in [csv_path, parquet_path, manifest_path]:
    assert path.exists(), f"Missing file: {path.name}"
    assert path.stat().st_size > 0, f"Empty file: {path.name}"

# 2. Read both exported datasets
df_csv_check = pd.read_csv(csv_path)
df_parquet_check = pd.read_parquet(parquet_path)

# 3. Validate dimensions
assert df_csv_check.shape == df_phase2_deliveries.shape
assert df_parquet_check.shape == df_phase2_deliveries.shape

# 4. Validate column names and order
assert df_csv_check.columns.tolist() == df_phase2_deliveries.columns.tolist()
assert df_parquet_check.columns.tolist() == df_phase2_deliveries.columns.tolist()

# 5. Validate delivery keys
delivery_key = ["match_id", "innings", "innings_ball_num"]

assert not df_csv_check.duplicated(subset=delivery_key).any()
assert not df_parquet_check.duplicated(subset=delivery_key).any()

assert (
    df_csv_check[delivery_key].to_numpy()
    == df_phase2_deliveries[delivery_key].to_numpy()
).all()

assert (
    df_parquet_check[delivery_key].to_numpy()
    == df_phase2_deliveries[delivery_key].to_numpy()
).all()

# 6. Validate critical classifications
critical_columns = [
    "legal_ball_evidence_class",
    "legal_ball_evidence_status",
    "legal_ball_final_treatment",
    "dismissal_type_normalized"
]

for col in critical_columns:
    assert df_csv_check[col].fillna("<NA>").equals(
        df_phase2_deliveries[col].fillna("<NA>").astype(str)
    ) or (
        df_csv_check[col].fillna("<NA>").astype(str).equals(
            df_phase2_deliveries[col].fillna("<NA>").astype(str)
        )
    )

    assert df_parquet_check[col].fillna("<NA>").astype(str).equals(
        df_phase2_deliveries[col].fillna("<NA>").astype(str)
    )

# 7. Verify manifest
df_manifest_check = pd.read_csv(manifest_path)

assert len(df_manifest_check) == 8
assert "phase2_integrated_cleaned_deliveries.csv" in (
    df_manifest_check["artifact"].values
)

print("CSV READ-BACK:", df_csv_check.shape)
print("PARQUET READ-BACK:", df_parquet_check.shape)
print("COLUMN ORDER: MATCHED")
print("DELIVERY KEYS: MATCHED")
print("CRITICAL CLASSIFICATIONS: MATCHED")
print("MANIFEST ENTRIES:", len(df_manifest_check))

print("\nPASS: Both exported datasets passed read-back verification.")

/tmp/ipykernel_21825/726266698.py:16: DtypeWarning: Columns (58) have mixed types. Specify dtype option on import or set low_memory=False.
  df_csv_check = pd.read_csv(csv_path)


CSV READ-BACK: (192235, 61)
PARQUET READ-BACK: (192235, 61)
COLUMN ORDER: MATCHED
DELIVERY KEYS: MATCHED
CRITICAL CLASSIFICATIONS: MATCHED
MANIFEST ENTRIES: 8

PASS: Both exported datasets passed read-back verification.


In [81]:
# CELL 16A — IDENTIFY MIXED-TYPE CSV COLUMN

print("COLUMN 58:", df_csv_check.columns[58])
print("IN-MEMORY DTYPE:",
      df_phase2_deliveries.iloc[:, 58].dtype)
print("CSV READ-BACK DTYPE:",
      df_csv_check.iloc[:, 58].dtype)

print("\nVALUE TYPE DISTRIBUTION IN ORIGINAL DATA")
display(
    df_phase2_deliveries.iloc[:, 58]
    .map(lambda x: type(x).__name__)
    .value_counts(dropna=False)
)

print("\nSAMPLE DISTINCT VALUES")
display(
    df_phase2_deliveries.iloc[:, 58]
    .dropna()
    .astype(str)
    .value_counts()
    .head(15)
)

COLUMN 58: season
IN-MEMORY DTYPE: object
CSV READ-BACK DTYPE: object

VALUE TYPE DISTRIBUTION IN ORIGINAL DATA


,count
season,
str,192235



SAMPLE DISTINCT VALUES


,count
season,
2025,43038
2024,40613
2022,39423
2026,32883
2023,31947
2022/23,2838
2024/25,754
2023/24,739


In [82]:
df_csv_season_check = pd.read_csv(
    csv_path,
    low_memory=False,
    dtype={"season": "string"}
)

print("SEASON DTYPE:", df_csv_season_check["season"].dtype)
print("MISSING SEASON VALUES:",
      df_csv_season_check["season"].isna().sum())

print("\nSEASON VALUE COUNTS")
display(
    df_csv_season_check["season"]
    .value_counts(dropna=False)
    .rename_axis("season")
    .reset_index(name="delivery_records")
)

assert len(df_csv_season_check) == 192235
assert df_csv_season_check["season"].notna().all()
assert (
    df_csv_season_check["season"].astype(str).to_numpy()
    == df_phase2_deliveries["season"].astype(str).to_numpy()
).all()

print("\nPASS: Season values preserved as strings in CSV export.")

SEASON DTYPE: string
MISSING SEASON VALUES: 0

SEASON VALUE COUNTS


,season,delivery_records
0,2025,43038
1,2024,40613
2,2022,39423
3,2026,32883
4,2023,31947
5,2022/23,2838
6,2024/25,754
7,2023/24,739



PASS: Season values preserved as strings in CSV export.


# Create the Phase 2 completion summary

In [83]:
from pathlib import Path
import pandas as pd

output_dir = Path("/content/phase2_outputs")

# Core population metrics
phase2_summary = [
    ("Target matches", len(df_target_matches)),
    ("Eligible matches", df_eligible_matches["match_id"].nunique()),
    ("Excluded matches", len(df_target_matches) - len(df_eligible_matches)),
    ("Eligible regulation deliveries", len(df_phase2_deliveries)),
    ("Super Over deliveries kept separate", len(df_eligible_super_over_deliveries)),
    ("Unique competitions", df_phase2_deliveries["comp_id"].nunique()),
    ("Integrated dataset columns", len(df_phase2_deliveries.columns)),
    ("Exception register entries", len(phase2_exceptions)),
    ("Unexplained extended overs", len(df_unexplained_over_full)),
    ("Unresolved legal-ball deliveries", (
        df_phase2_deliveries["legal_ball_final_treatment"] == "UNRESOLVED"
    ).sum()),
    ("Explicit wide/overstep records", (
        df_phase2_deliveries["legal_ball_final_treatment"] == "NON_LEGAL_EVIDENCE"
    ).sum()),
    ("Delivery key duplicates", df_phase2_deliveries.duplicated(
        ["match_id", "innings", "innings_ball_num"]
    ).sum()),
    ("Full-row duplicates", df_phase2_deliveries.duplicated().sum()),
    ("Unmatched team identity exceptions", len(df_team_identity_exceptions))
]

df_phase2_completion_summary = pd.DataFrame(
    phase2_summary,
    columns=["metric", "value"]
)

print("PHASE 2 COMPLETION SUMMARY")
display(df_phase2_completion_summary)

# Verify required export artifacts
required_files = [
    "phase2_integrated_cleaned_deliveries.csv",
    "phase2_integrated_cleaned_deliveries.parquet",
    "phase2_exception_register.csv",
    "phase2_legal_ball_evidence.csv",
    "phase2_unexplained_extended_over_register.csv",
    "phase2_dismissal_classified.csv",
    "phase2_team_identity_audit.csv",
    "phase2_team_identity_exceptions.csv",
    "phase2_over_sequence_audit.csv",
    "phase2_export_manifest.csv"
]

file_check = pd.DataFrame([
    {
        "artifact": filename,
        "exists": (output_dir / filename).exists(),
        "size_bytes": (
            (output_dir / filename).stat().st_size
            if (output_dir / filename).exists()
            else 0
        )
    }
    for filename in required_files
])

print("\nEXPORT ARTIFACT CHECK")
display(file_check)

assert file_check["exists"].all()
assert (file_check["size_bytes"] > 0).all()
assert len(df_phase2_deliveries) == 192235
assert len(phase2_exceptions) == 24
assert df_phase2_deliveries["match_id"].nunique() == 813

summary_path = output_dir / "phase2_completion_summary.csv"
df_phase2_completion_summary.to_csv(summary_path, index=False)

print("\nPASS: Phase 2 completion summary created.")
print("Saved:", summary_path)

PHASE 2 COMPLETION SUMMARY


,metric,value
0,Target matches,853
1,Eligible matches,813
2,Excluded matches,40
3,Eligible regulation deliveries,192235
4,Super Over deliveries kept separate,26
5,Unique competitions,136
6,Integrated dataset columns,61
7,Exception register entries,24
8,Unexplained extended overs,147
9,Unresolved legal-ball deliveries,257



EXPORT ARTIFACT CHECK


,artifact,exists,size_bytes
0,phase2_integrated_cleaned_deliveries.csv,True,90137196
1,phase2_integrated_cleaned_deliveries.parquet,True,3456892
2,phase2_exception_register.csv,True,6829
3,phase2_legal_ball_evidence.csv,True,53412266
4,phase2_unexplained_extended_over_register.csv,True,4500
5,phase2_dismissal_classified.csv,True,39264086
6,phase2_team_identity_audit.csv,True,295
7,phase2_team_identity_exceptions.csv,True,1
8,phase2_over_sequence_audit.csv,True,919553
9,phase2_export_manifest.csv,True,578



PASS: Phase 2 completion summary created.
Saved: /content/phase2_outputs/phase2_completion_summary.csv
